# DNA-binding model line

A **separate model line for DNA**, not a variant of the nuclear model: the target is "dna binding residue", the
models are trained on it, selected on it, and reported on it.

**Part 1 is notebook 09 unchanged** — every state is cached on Drive, so it only reloads (no retraining) and rebuilds
`C3_union`, the current nuclear final.

**Part 2 (`LX-`) is this line.** The per-residue nuclear labels are never changed. BioLiP2 (snapshot 2026-09-29,
resolution ≤ 3 Å) only says which nucleic acid each protein binds, so the target follows from labels we already had:

| protein | y |
|---|---|
| binds dna (or both) | its nuclear labels |
| binds the other nucleic acid | 0 |
| no nucleic-acid site | 0 |
| not found in BioLiP (6 proteins in Dev, none in the tests) | **masked**: dropped from the loss and from every metric |

| set | DNA positives | proteins with DNA sites |
|---|---|---|
| DevSet1010 | 1217 | 62 |
| TestSet300 | 1307 | 57 |
| TestSetNew46 | 48 | 4 |

TestSet300 is dominated by DNA: 1307 of its 1470 nucleic-acid positive residues are DNA (the 32 residues of the one protein that binds both are counted in each line), so this is the line where a gain shows up most clearly on the tests.

| cell | what | decision |
|---|---|---|
| LX-1 | build the target, mask the unknown proteins, masked folds | — |
| LX-2 | **the model to beat**: the current nuclear final scored against this target | — |
| LX-3 | V1 + MS hypergraphs (ESM + ACCESS + patches + ESM-IF + ProtT5) trained on this target, 3 seeds each, masked loss | — |
| LX-4 | candidates: nuclear final / the pair / the pair + nuclear final | **cross-fitted** Dev, margin 0.002 |
| LX-5 | external AUPRC, precision and recall at the Dev-OOF threshold, bootstrap B = 10000 vs the nuclear final | Dev winner stands unless the pooled-test CI95 is entirely below 0 |
| LX-6 | zip → `EXPORT/nuclear_dna_line.zip` | — |

Honest about the size: Dev has 62 proteins with DNA sites. The protocol (Dev-only choice, cross-fitted weights,
0.002 margin, bootstrap) is what keeps a lucky split from being adopted.

**How to run:** Runtime → GPU (T4) → **Run all**. Part 1 ≈ 20–40 min of reloading, then 6 new 5-fold runs.
Resume-safe: every fold is saved on Drive, so **Run all** again after a disconnect.


In [ ]:
# ============================================================
# 0. DRIVE / INSTALL / IMPORT
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, math, shutil, subprocess, platform
from collections import Counter, defaultdict

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "pandas", "scipy", "scikit-learn",
     "biopython", "tqdm"],
    check=True
)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset
from scipy.spatial import cKDTree
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, matthews_corrcoef,
    precision_score, recall_score
)
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from Bio.SeqUtils import seq1
from copy import deepcopy

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
ULTIMATE_CANDIDATES = [
    BASE/'GEAR_ULTIMATE_OTHER_BINDINGS',
    BASE/'GEAR_ULTIMATE',
]
ULTIMATE_DIR = next(
    (p for p in ULTIMATE_CANDIDATES if (p/'gear_ultimate').exists()),
    None
)

if ULTIMATE_DIR is None:
    raise FileNotFoundError(
        'GEAR backend not found under GEAR_ULTIMATE_OTHER_BINDINGS or GEAR_ULTIMATE.'
    )

req = ULTIMATE_DIR/'requirements.txt'
if req.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True
    )

sys.path.insert(0, str(ULTIMATE_DIR))

try:
    from gear_ultimate import GEAR_ULTIMATE_VERSION as GEAR_VERSION
except Exception:
    from gear_ultimate import __version__ as GEAR_VERSION

from gear_ultimate import gear_core as core
from gear_ultimate import gear_structure as structure
from gear_ultimate import gear_mechanistic as mech
from gear_ultimate import gear_graph as graph
from gear_ultimate import gear_evolution as evo
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')

RUN_ROOT = BASE/'results'/'NUCLEAR_ULTIMATE_V1_DATA_DRIVEN_PATCH'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = BASE/'cache'/'nuclear_ultimate_v1'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print('BASE:', BASE)
print('GEAR backend:', ULTIMATE_DIR)
print('GEAR version:', GEAR_VERSION)
print('Device:', DEVICE)
print('Run root:', RUN_ROOT)


## 1. Master switches

Default = full nuclear experiment.

All expensive generated blocks and all custom model folds are persisted and resume-safe.


In [ ]:
# ============================================================
# 1. MASTER SWITCHES
# ============================================================
ESM_BATCH_SIZE = 8

RUN_FEATURE_GENERATION = True
RUN_FAMILY_SCREEN = True
RUN_COMBINATION_LADDER = True
RUN_CUSTOM_ARCHITECTURES = True
RUN_PATCH_HYPERGRAPH = True
RUN_FINAL_EXTERNAL = True

ATTACH_EVO_IF_CACHED = True
ATTACH_ESMIF_IF_CACHED = False   # [V3-FIX] keep the V1 screens identical; V3 adds ESM-IF itself
FORCE_REBUILD_NA_FEATURES = False
FORCE_REBUILD_PATCHES = False

MIN_MEAN_AUPRC_GAIN = 0.002
GLOBAL_COMPLEXITY_MARGIN = 0.002

CUSTOM_BATCH = 4096
CUSTOM_EPOCHS = 45
CUSTOM_PATIENCE = 7
CUSTOM_LR = 3e-4
CUSTOM_WEIGHT_DECAY = 1e-4
SEED = 2026

PATCH_RADIUS_A = 10.0
PATCH_MAX_PER_PROTEIN = 128
PATCH_PROTEIN_BATCH = 4
PATCH_EPOCHS = 32
PATCH_PATIENCE = 6
PATCH_LR = 3e-4
PATCH_WEIGHT_DECAY = 1e-4
PATCH_LATENT = 192

print('Full run:', {
    'feature_generation': RUN_FEATURE_GENERATION,
    'family_screen': RUN_FAMILY_SCREEN,
    'combination_ladder': RUN_COMBINATION_LADDER,
    'custom_architectures': RUN_CUSTOM_ARCHITECTURES,
    'patch_hypergraph': RUN_PATCH_HYPERGRAPH,
    'external': RUN_FINAL_EXTERNAL,
})


In [ ]:
# ============================================================
# 2. PREPARE / REUSE NUCLEAR GEAR CONTEXT
# ============================================================
ctx = suite.prepare_context(
    base=BASE,
    run_version='nuclear_ultimate_v1_context',
    label_column='nuclear_binding',
    models=('MLP',),
    ensure_base=True,
    add_mech=True,
    add_multi=True,
    add_seqctx=True,
    esm_batch_size=ESM_BATCH_SIZE,
    device=DEVICE,
)

display(suite.audit_context(ctx))
display(reporting.dimension_table(ctx.stores['train']))

# Existing GEAR stores must already be exact DevSet1010 / Test300 / Test46.
assert len(ctx.stores['train']) == 1010
assert len(ctx.stores['test300']) == 300
assert len(ctx.stores['test46']) == 46

print('Context sizes: PASS')


In [ ]:
# ============================================================
# 3. DATAFRAME / OFFICIAL-FOLD AUDIT
# ============================================================
TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_df(path):
    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)

    for c in [
        'metal_binding',
        'nuclear_binding',
        'small_binding',
        'any_ligand_binding_sites',
    ]:
        if c in df.columns:
            df[c] = df[c].apply(parse_list)

    for r in df.itertuples():
        if len(r.sequence) != len(r.nuclear_binding):
            raise RuntimeError(f'{r.prot_id}: nuclear label/sequence mismatch')

    return df

raw_train = load_df(TRAIN_CSV)
test300_df = load_df(TEST300_CSV)
test46_df = load_df(TEST46_CSV)

BASELINE_WORK = BASE/'baseline_reproduction_master_v6'
BN24_CANDIDATES = [
    BASELINE_WORK/'repos'/'bindnode24',
    BASE/'repos'/'bindnode24',
]

BN24 = next((p for p in BN24_CANDIDATES if (p/'.git').exists()), None)

if BN24 is None:
    BN24 = Path('/content/bindnode24_nuclear_v1')
    if not BN24.exists():
        subprocess.run(
            ['git','clone','-q','https://github.com/erckert/bindNode24.git',str(BN24)],
            check=True
        )

fold_root = BN24/'dataset'/'cv_splits'/'predicted_structures'
OFFICIAL_FOLDS = {}
official_order = []
seen = set()

for k in range(1,6):
    ids = [
        x.strip()
        for x in (fold_root/f'ids_split{k}.txt').read_text().splitlines()
        if x.strip()
    ]
    if seen.intersection(ids):
        raise RuntimeError(f'Fold overlap at fold {k}')
    OFFICIAL_FOLDS[k] = ids
    official_order.extend(ids)
    seen.update(ids)

train_lookup = raw_train.set_index('prot_id', drop=False)
missing = set(official_order) - set(raw_train.prot_id)
if missing:
    raise RuntimeError(f'Official Dev IDs missing from project CSV: {sorted(missing)[:20]}')

train_df = train_lookup.loc[official_order].reset_index(drop=True)

assert len(train_df) == 1010
assert len(test300_df) == 300
assert len(test46_df) == 46

SPLIT_DF = {
    'DevSet1010': train_df,
    'TestSet300': test300_df,
    'TestSetNew46': test46_df,
}
STORE_KEY = {
    'DevSet1010':'train',
    'TestSet300':'test300',
    'TestSetNew46':'test46',
}

# Cross-check store IDs.
for split, df in SPLIT_DF.items():
    store_ids = set(map(str, ctx.stores[STORE_KEY[split]].keys()))
    csv_ids = set(df.prot_id.astype(str))
    if store_ids != csv_ids:
        raise RuntimeError(
            f'{split}: store/CSV mismatch. '
            f'missing_in_store={sorted(csv_ids-store_ids)[:10]} '
            f'extra_in_store={sorted(store_ids-csv_ids)[:10]}'
        )

print('Exact dataset/fold alignment: PASS')
print('Fold sizes:', {k:len(v) for k,v in OFFICIAL_FOLDS.items()})


## 4. Reuse orthogonal biological context where available

`EVO48` is retained because the previous nuclear experiments showed a modest but reproducible Dev gain.

ESM-IF is **not required** for V1. If a complete per-residue ESM-IF cache already exists from earlier runs, it is attached as an optional structural control; otherwise V1 proceeds without generating a new 1010-protein ESM-IF cache.


In [ ]:
# ============================================================
# 4. ATTACH EVO / OPTIONAL EXISTING ESMIF
# ============================================================
EVO_AVAILABLE = False

if ATTACH_EVO_IF_CACHED:
    try:
        coverage = evo.attach_evo(
            ctx.stores,
            BASE,
            splits=('train','test300','test46'),
            require_msa=True,
        )
        display(coverage)
        EVO_AVAILABLE = True
        print('EVO48 attached.')
    except Exception as e:
        print('EVO cache unavailable/incomplete -> skip EVO:', repr(e))

# Robust store helpers.
def get_record(store, pid):
    pid = str(pid)
    if pid in store:
        return store[pid]
    raise KeyError(pid)

def get_block(store, pid, block):
    rec = get_record(store, pid)

    if isinstance(rec, dict):
        if block in rec:
            return np.asarray(rec[block])
        if isinstance(rec.get('features'), dict) and block in rec['features']:
            return np.asarray(rec['features'][block])

    if hasattr(rec, block):
        return np.asarray(getattr(rec, block))

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec, attr, None)
        if isinstance(d, dict) and block in d:
            return np.asarray(d[block])

    raise KeyError(f'{pid}/{block}')

def set_block(store, pid, block, arr):
    rec = get_record(store, pid)
    arr = np.asarray(arr, dtype=np.float32)

    if isinstance(rec, dict):
        if isinstance(rec.get('features'), dict):
            rec['features'][block] = arr
        else:
            rec[block] = arr
        return

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec, attr, None)
        if isinstance(d, dict):
            d[block] = arr
            return

    setattr(rec, block, arr)

# Optional reuse-only ESMIF.
ESMIF_AVAILABLE = False
ESMIF_CANDIDATE_ROOTS = [
    BASE/'cache'/'gear_metal_v3'/'esmif_v1',
    BASE/'cache'/'nuclear_ultimate_v1'/'esmif_v1',
]

if ATTACH_ESMIF_IF_CACHED:
    for root in ESMIF_CANDIDATE_ROOTS:
        split_dirs = {
            'DevSet1010': root/'DevSet1010'/'esm_if',
            'TestSet300': root/'TestSet300'/'esm_if',
            'TestSetNew46': root/'TestSetNew46'/'esm_if',
        }

        if not all(d.exists() for d in split_dirs.values()):
            continue

        good = True
        for split, df in SPLIT_DF.items():
            d = split_dirs[split]
            for r in df.itertuples():
                fp = d/f'{r.prot_id}.npy'
                if not fp.exists():
                    good = False
                    break
                a = np.load(fp, mmap_mode='r')
                if a.shape != (len(r.sequence),512):
                    good = False
                    break
            if not good:
                break

        if good:
            for split, df in SPLIT_DF.items():
                store = ctx.stores[STORE_KEY[split]]
                d = split_dirs[split]
                for r in df.itertuples():
                    set_block(
                        store,
                        r.prot_id,
                        'ESMIF',
                        np.load(d/f'{r.prot_id}.npy').astype(np.float32)
                    )

            ESMIF_AVAILABLE = True
            print('Reused complete ESMIF cache:', root)
            break

print('EVO_AVAILABLE  =', EVO_AVAILABLE)
print('ESMIF_AVAILABLE=', ESMIF_AVAILABLE)


# Part A — Nuclear-specific feature generation

The new blocks are intentionally different from the metal ontology.

## `NA_SEQELECTRO32`
Interpretable sequence-local cationic/electrostatic environment.

## `NA_PATCH3D40`
3D neighborhood composition and electrostatic proxy around each residue.

## `NA_HBOND16`
Cationic + donor/acceptor chemistry and local H-bond-capable environment.

## `NA_TOPO12`
Topology/connectedness of spatial basic/polar neighborhoods.

## `NA_PROTCTX8`
Soft protein-level physicochemical context. This is a control/conditioning feature, never a hard gate.

## `NA_AROMCTRL8`
Aromatic/stacking-oriented control. Audit data does **not** support making aromatic chemistry a core nuclear assumption, so this block must earn its place on Dev.


In [ ]:
# ============================================================
# 5. SHARED EXACT PDB CACHE — NUCLEAR + SMALL
# ============================================================
from pathlib import Path
import shutil
import tarfile
import json
import hashlib
import os

AA3 = {
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
}

# ------------------------------------------------------------
# ONE local cache shared by nuclear + small
# ------------------------------------------------------------
LOCAL_SHARED_ROOT = Path('/content/shared_exact_pdb_v1')

DRIVE_ARCHIVE = (
    BASE / 'cache' / 'shared_exact_pdb_v1.tar.gz'
)

DRIVE_ARCHIVE.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Downstream cells continue using PDB_NORM_ROOT.
PDB_NORM_ROOT = LOCAL_SHARED_ROOT

# Existing work that we should salvage rather than redo.
OLD_NORMALIZED_ROOTS = [
    BASE / 'cache' / 'nuclear_ultimate_v1' / 'normalized_pdb',
    BASE / 'cache' / 'small_ultimate_v1' / 'normalized_pdb',

    BASELINE_WORK
    / 'cache'
    / 'metal_external_normalized_structures',
]

# Raw source roots only used when no normalized copy exists.
RAW_ROOTS = [
    BASE / 'esmFold_pdb_files',
    BASE / 'experimental_pdb',
    BN24 / 'dataset' / 'AlphaFold_predictions',
]

_parser = PDBParser(QUIET=True)


# ============================================================
# helpers
# ============================================================
def standard_residues(chain):
    return [
        r for r in chain
        if (
            r.id[0] == ' '
            and r.resname.strip().upper() in AA3
        )
    ]


def residue_sequence(rr):
    return ''.join(
        seq1(r.resname.strip().upper())
        for r in rr
    )


def pdb_exact_ok(path, target_seq):
    """
    Check whether PDB is already exact chain-A normalized.

    Important:
    use this mainly on /content or on a candidate file,
    NOT recursively over Drive.
    """
    path = Path(path)

    if not path.exists():
        return False

    try:
        st = _parser.get_structure(
            'check',
            str(path)
        )

        model = next(st.get_models())

        if 'A' not in model:
            return False

        rr = standard_residues(
            model['A']
        )

        return (
            residue_sequence(rr)
            == target_seq
        )

    except Exception:
        return False


def normalize_exact_chain_A(src, target_seq, dst):
    """
    Normalize ONLY if we truly do not already have
    an exact-normalized copy.
    """
    st = _parser.get_structure(
        'src',
        str(src)
    )

    model = next(
        st.get_models()
    )

    exact = []
    diag = []

    for ch in model:
        rr = standard_residues(ch)
        seq = residue_sequence(rr)

        diag.append(
            (
                str(ch.id),
                len(seq)
            )
        )

        if seq == target_seq:
            exact.append(
                (
                    ch,
                    rr
                )
            )

    if not exact:
        raise RuntimeError(
            f'{src}: no exact chain match; '
            f'target={len(target_seq)}, '
            f'chains={diag}'
        )

    exact.sort(
        key=lambda x: (
            0
            if str(x[0].id) == 'A'
            else 1,

            str(x[0].id)
        )
    )

    _, rr = exact[0]

    out_st = Structure('normalized')
    out_model = Model(0)
    out_chain = Chain('A')

    out_st.add(out_model)
    out_model.add(out_chain)

    for i, res in enumerate(
        rr,
        start=1
    ):
        r2 = deepcopy(res)
        r2.detach_parent()

        r2.id = (
            ' ',
            i,
            ' '
        )

        out_chain.add(
            r2
        )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    io = PDBIO()
    io.set_structure(
        out_st
    )

    io.save(
        str(dst)
    )


# ============================================================
# 1. RESTORE SHARED CACHE IF IT EXISTS
# ============================================================
if (
    DRIVE_ARCHIVE.exists()
    and not LOCAL_SHARED_ROOT.exists()
):
    print(
        'Found shared PDB archive.'
    )

    print(
        'Copy Drive -> /content ...'
    )

    local_tar = (
        Path('/content')
        / DRIVE_ARCHIVE.name
    )

    shutil.copy2(
        DRIVE_ARCHIVE,
        local_tar
    )

    print(
        'Extracting locally ...'
    )

    with tarfile.open(
        local_tar,
        'r:gz'
    ) as tf:
        tf.extractall(
            '/content'
        )

    local_tar.unlink(
        missing_ok=True
    )


LOCAL_SHARED_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. FAST LOOKUP OF OLD NORMALIZED STRUCTURES
# ============================================================
# Direct paths first — no rglob.
def normalized_candidates(
    split,
    pid
):
    pid = str(pid)

    aliases = {
        'DevSet1010': [
            'DevSet1010',
            'train',
        ],

        'TestSet300': [
            'TestSet300',
            'test300',
        ],

        'TestSetNew46': [
            'TestSetNew46',
            'test46',
        ],
    }

    out = []

    for root in OLD_NORMALIZED_ROOTS:
        out.append(
            root / f'{pid}.pdb'
        )

        for alias in aliases.get(
            split,
            []
        ):
            out.append(
                root
                / alias
                / f'{pid}.pdb'
            )

    return out


def raw_candidates(
    pid
):
    pid = str(pid)

    return [
        BASE
        / 'esmFold_pdb_files'
        / f'{pid}.pdb',

        BASE
        / 'experimental_pdb'
        / f'{pid}.pdb',

        BN24
        / 'dataset'
        / 'AlphaFold_predictions'
        / f'{pid}.pdb',
    ]


# Only build fallback index if direct paths actually fail.
RAW_FALLBACK_INDEX = None


def get_raw_fallback_index():
    global RAW_FALLBACK_INDEX

    if RAW_FALLBACK_INDEX is not None:
        return RAW_FALLBACK_INDEX

    print(
        '\nDirect raw paths missed at least one protein.'
    )

    print(
        'Building ONE fallback source index...'
    )

    idx = {}

    for root in RAW_ROOTS:
        if not root.exists():
            continue

        print(
            ' scanning:',
            root
        )

        for dirpath, _, filenames in os.walk(
            root
        ):
            for fn in filenames:
                if not fn.endswith(
                    '.pdb'
                ):
                    continue

                p = (
                    Path(dirpath)
                    / fn
                )

                idx.setdefault(
                    p.stem,
                    p
                )

    RAW_FALLBACK_INDEX = idx

    print(
        'fallback PDBs indexed:',
        len(idx)
    )

    return idx


# ============================================================
# 3. BUILD SHARED LOCAL CACHE
# ============================================================
stats = {
    'already_local': 0,
    'reused_old_normalized': 0,
    'normalized_raw': 0,
}

missing = []

TOTAL = sum(
    len(df)
    for df in SPLIT_DF.values()
)

done = 0


for split, df in SPLIT_DF.items():

    outdir = (
        LOCAL_SHARED_ROOT
        / split
    )

    outdir.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        '\n',
        split,
        len(df),
        'proteins'
    )

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        dst = (
            outdir
            / f'{pid}.pdb'
        )

        # ----------------------------------
        # A. already local shared
        # ----------------------------------
        if pdb_exact_ok(
            dst,
            r.sequence
        ):
            stats[
                'already_local'
            ] += 1

            done += 1
            continue


        # ----------------------------------
        # B. reuse nuclear/small/metal
        #    normalized PDB AS-IS
        # ----------------------------------
        reused = False

        for cand in normalized_candidates(
            split,
            pid
        ):

            if not cand.exists():
                continue

            if pdb_exact_ok(
                cand,
                r.sequence
            ):
                shutil.copy2(
                    cand,
                    dst
                )

                stats[
                    'reused_old_normalized'
                ] += 1

                reused = True
                break


        if reused:
            done += 1

            if done % 100 == 0:
                print(
                    f'  {done}/{TOTAL}',
                    stats
                )

            continue


        # ----------------------------------
        # C. raw source direct lookup
        # ----------------------------------
        src = None

        for cand in raw_candidates(
            pid
        ):
            if cand.exists():
                src = cand
                break


        # ----------------------------------
        # D. last-resort ONE-TIME index
        # ----------------------------------
        if src is None:
            src = get_raw_fallback_index().get(
                pid
            )


        if src is None:
            missing.append(
                (
                    split,
                    pid
                )
            )

            continue


        # ----------------------------------
        # E. normalize only genuinely missing
        # ----------------------------------
        normalize_exact_chain_A(
            src,
            r.sequence,
            dst
        )

        if not pdb_exact_ok(
            dst,
            r.sequence
        ):
            raise RuntimeError(
                f'{split}/{pid}: '
                'normalized PDB failed validation'
            )

        stats[
            'normalized_raw'
        ] += 1

        done += 1


        if done % 100 == 0:
            print(
                f'  {done}/{TOTAL}',
                stats
            )


if missing:
    raise FileNotFoundError(
        f'Missing source PDB for '
        f'{len(missing)} proteins: '
        f'{missing[:20]}'
    )


print(
    '\nCACHE BUILD COMPLETE'
)

print(
    stats
)


# ============================================================
# 4. SAVE ONE ARCHIVE TO DRIVE
# ============================================================
print(
    '\nPacking shared local cache...'
)

local_tar = (
    Path('/content')
    / 'shared_exact_pdb_v1.tar.gz'
)

with tarfile.open(
    local_tar,
    'w:gz'
) as tf:

    tf.add(
        LOCAL_SHARED_ROOT,
        arcname='shared_exact_pdb_v1'
    )


print(
    'Copying ONE archive to Drive...'
)

shutil.copy2(
    local_tar,
    DRIVE_ARCHIVE
)

local_tar.unlink(
    missing_ok=True
)


print(
    '\nExact-sequence shared structures: PASS'
)

print(
    'PDB_NORM_ROOT =',
    PDB_NORM_ROOT
)

print(
    'Persistent archive =',
    DRIVE_ARCHIVE
)

In [ ]:
# ============================================================
# 6. NUCLEAR-SPECIFIC FEATURE DEFINITIONS
# ============================================================
FEATURE_CACHE = CACHE_ROOT/'nuclear_explicit_v1'
FEATURE_CACHE.mkdir(parents=True, exist_ok=True)

CHARGE = {
    'K':1.0,
    'R':1.0,
    'H':0.10,
    'D':-1.0,
    'E':-1.0,
}

GROUPS = {
    'KR': set('KR'),
    'KRH': set('KRH'),
    'DE': set('DE'),
    'POLAR': set('NQSTY'),
    'AROM': set('FWYH'),
    'HYDRO': set('AVILMFWY'),
    'HBD': set('KRHNQSTYW'),
    'HBA': set('DEHNQSTYC'),
}

BACKBONE_ATOMS = {'N','CA','C','O','OXT'}

def representative_coord(res):
    side = [
        np.asarray(a.coord,dtype=np.float32)
        for a in res
        if a.element != 'H' and a.name not in BACKBONE_ATOMS
    ]
    if side:
        return np.mean(np.stack(side),axis=0)

    if 'CA' in res:
        return np.asarray(res['CA'].coord,dtype=np.float32)

    atoms = [np.asarray(a.coord,dtype=np.float32) for a in res]
    if not atoms:
        raise RuntimeError('Residue has no atoms')
    return np.mean(np.stack(atoms),axis=0)

def sequence_block(seq):
    L = len(seq)
    out = np.zeros((L,32),dtype=np.float32)

    q = np.array([CHARGE.get(a,0.0) for a in seq],dtype=np.float32)

    for i,a in enumerate(seq):
        out[i,0] = q[i]
        out[i,1] = float(a=='K')
        out[i,2] = float(a=='R')
        out[i,3] = float(a=='H')
        out[i,4] = float(a in GROUPS['DE'])
        out[i,5] = float(a in GROUPS['POLAR'])
        out[i,6] = float(a in GROUPS['AROM'])
        out[i,7] = float(a in GROUPS['HYDRO'])

        col = 8
        for radius in [2,4,7,10]:
            lo=max(0,i-radius)
            hi=min(L,i+radius+1)
            w=seq[lo:hi]
            qq=q[lo:hi]

            vals = [
                sum(x in GROUPS['KR'] for x in w)/len(w),
                sum(x in GROUPS['KRH'] for x in w)/len(w),
                sum(x in GROUPS['DE'] for x in w)/len(w),
                sum(x in GROUPS['POLAR'] for x in w)/len(w),
                float(np.mean(qq)),
                sum(x in GROUPS['HYDRO'] for x in w)/len(w),
            ]

            out[i,col:col+6] = vals
            col += 6

    return out

def union_find_components(indices, coords, threshold=8.0):
    if len(indices)==0:
        return {}

    idx=np.asarray(indices,dtype=int)
    cc=np.asarray(coords,dtype=np.float32)

    parent=list(range(len(idx)))

    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]
            x=parent[x]
        return x

    def union(a,b):
        ra,rb=find(a),find(b)
        if ra!=rb:
            parent[rb]=ra

    if len(idx)>1:
        tree=cKDTree(cc)
        for a,b in tree.query_pairs(threshold):
            union(a,b)

    comp=defaultdict(list)
    for j,res_idx in enumerate(idx):
        comp[find(j)].append(int(res_idx))

    size={}
    for members in comp.values():
        s=len(members)
        for r in members:
            size[r]=s

    return size

def build_3d_blocks(pdb_path, seq):
    st=PDBParser(QUIET=True).get_structure('p',str(pdb_path))
    m=next(st.get_models())
    residues=standard_residues(m['A'])

    if residue_sequence(residues)!=seq:
        raise RuntimeError(f'{pdb_path}: sequence mismatch')

    coords=np.stack(
        [representative_coord(r) for r in residues],
        axis=0
    ).astype(np.float32)

    L=len(seq)
    tree=cKDTree(coords)
    q=np.array([CHARGE.get(a,0.0) for a in seq],dtype=np.float32)

    basic_idx=[i for i,a in enumerate(seq) if a in GROUPS['KRH']]
    basic_comp=union_find_components(
        basic_idx,
        coords[basic_idx] if basic_idx else np.zeros((0,3),dtype=np.float32),
        threshold=8.0
    )

    patch=np.zeros((L,40),dtype=np.float32)
    hbond=np.zeros((L,16),dtype=np.float32)
    topo=np.zeros((L,12),dtype=np.float32)
    arom=np.zeros((L,8),dtype=np.float32)

    # Global spatial adjacency for topology.
    neigh8=[tree.query_ball_point(coords[i],8.0) for i in range(L)]
    neigh10=[tree.query_ball_point(coords[i],10.0) for i in range(L)]

    for i,a in enumerate(seq):
        # -------- own H-bond chemistry --------
        hbond[i,0]=float(a in GROUPS['HBD'])
        hbond[i,1]=float(a in GROUPS['HBA'])
        hbond[i,2]=float(a=='R')                 # guanidinium
        hbond[i,3]=float(a=='K')                 # amine
        hbond[i,4]=float(a=='H')                 # imidazole
        hbond[i,5]=float(a in set('NQ'))         # amide
        hbond[i,6]=float(a in set('STY'))        # hydroxyl
        hbond[i,7]=float(a in set('DE'))         # carboxylate

        # -------- 3D radius summaries --------
        col=0

        for radius in [6.0,8.0,10.0,12.0]:
            ids=np.asarray(tree.query_ball_point(coords[i],radius),dtype=int)
            ids=ids[ids!=i]

            if len(ids):
                aa=[seq[j] for j in ids]
                d=np.linalg.norm(coords[ids]-coords[i][None,:],axis=1)
                qn=q[ids]

                vals=[
                    min(len(ids),40)/40.0,
                    sum(x in GROUPS['KR'] for x in aa)/len(aa),
                    sum(x in GROUPS['KRH'] for x in aa)/len(aa),
                    sum(x in GROUPS['DE'] for x in aa)/len(aa),
                    sum(x in GROUPS['POLAR'] for x in aa)/len(aa),
                    sum(x in GROUPS['HYDRO'] for x in aa)/len(aa),
                    float(np.mean(qn)),
                    float(np.tanh(np.sum(qn/(d+1.0)))),
                ]
            else:
                vals=[0.0]*8

            patch[i,col:col+8]=vals
            col+=8

        # -------- local H-bond-capable environment --------
        col=8

        for radius in [6.0,10.0]:
            ids=np.asarray(tree.query_ball_point(coords[i],radius),dtype=int)
            ids=ids[ids!=i]
            aa=[seq[j] for j in ids]

            den=max(len(aa),1)

            hbond[i,col:col+4]=[
                sum(x in GROUPS['HBD'] for x in aa)/den,
                sum(x in GROUPS['HBA'] for x in aa)/den,
                sum(x in GROUPS['KRH'] for x in aa)/den,
                sum(x in GROUPS['POLAR'] for x in aa)/den,
            ]
            col+=4

        # -------- nearest chemistry distances --------
        def nearest(chars):
            ids=[j for j,x in enumerate(seq) if x in chars and j!=i]
            if not ids:
                return 1.0
            d=np.linalg.norm(coords[ids]-coords[i][None,:],axis=1)
            return min(float(d.min())/20.0,1.0)

        patch[i,32]=nearest(GROUPS['KR'])
        patch[i,33]=nearest(GROUPS['DE'])
        patch[i,34]=nearest(GROUPS['POLAR'])
        patch[i,35]=nearest(GROUPS['AROM'])

        # -------- higher-order basic component proximity --------
        basic_near=[
            j for j in tree.query_ball_point(coords[i],10.0)
            if j in basic_comp
        ]
        max_comp=max(
            [basic_comp[j] for j in basic_near],
            default=0
        )
        patch[i,36]=min(max_comp,20)/20.0

        def centroid_distance(chars):
            ids=[j for j,x in enumerate(seq) if x in chars and
                 np.linalg.norm(coords[j]-coords[i])<=10.0]
            if not ids:
                return 1.0
            c=coords[ids].mean(axis=0)
            return min(float(np.linalg.norm(coords[i]-c))/20.0,1.0)

        patch[i,37]=centroid_distance(GROUPS['KRH'])
        patch[i,38]=centroid_distance(GROUPS['POLAR'])

        ids=np.asarray(neigh10[i],dtype=int)
        ids=ids[ids!=i]
        if len(ids):
            patch[i,39]=min(
                float(np.mean(np.linalg.norm(
                    coords[ids]-coords[i][None,:],
                    axis=1
                )))/20.0,
                1.0
            )

        # -------- topology --------
        n8=[j for j in neigh8[i] if j!=i]
        n10=[j for j in neigh10[i] if j!=i]

        topo[i,0]=min(len(n8),30)/30.0
        topo[i,1]=min(len(n10),40)/40.0

        b8=[j for j in n8 if seq[j] in GROUPS['KRH']]
        p8=[j for j in n8 if seq[j] in GROUPS['POLAR']]
        d8=[j for j in n8 if seq[j] in GROUPS['DE']]

        topo[i,2]=min(len(b8),15)/15.0
        topo[i,3]=min(len(p8),20)/20.0
        topo[i,4]=min(len(d8),15)/15.0
        topo[i,5]=np.tanh((len(b8)-len(d8))/5.0)
        topo[i,6]=min(max_comp,20)/20.0
        topo[i,7]=min(len(b8)+len(d8),20)/20.0

        if n8:
            seqsep=np.abs(np.asarray(n8)-i)
            topo[i,8]=float(np.mean(seqsep>10))
            topo[i,9]=min(float(np.mean(seqsep))/50.0,1.0)

            # Local clustering coefficient among spatial neighbors.
            if len(n8)>=2:
                cc=coords[n8]
                t2=cKDTree(cc)
                n_edges=len(t2.query_pairs(8.0))
                possible=len(n8)*(len(n8)-1)/2
                topo[i,10]=n_edges/max(possible,1)

        if b8:
            bd=np.linalg.norm(coords[b8]-coords[i][None,:],axis=1)
            topo[i,11]=min(
                (len(b8)/(1.0+float(np.mean(bd))))/3.0,
                1.0
            )

        # -------- aromatic control --------
        arom[i,0]=float(a in GROUPS['AROM'])
        arom[i,1]=float(a in set('YH'))

        for col2,radius in zip([2,3,4],[6.0,8.0,10.0]):
            ids=[
                j for j in tree.query_ball_point(coords[i],radius)
                if j!=i
            ]
            arom[i,col2]=(
                sum(seq[j] in GROUPS['AROM'] for j in ids)/max(len(ids),1)
            )

        for col2,rad in zip([5,6],[4,10]):
            lo=max(0,i-rad)
            hi=min(L,i+rad+1)
            w=seq[lo:hi]
            arom[i,col2]=sum(x in GROUPS['AROM'] for x in w)/len(w)

        arom[i,7]=nearest(GROUPS['AROM'])

    return patch,hbond,topo,arom

def protein_context(seq):
    L=len(seq)
    q=np.asarray([CHARGE.get(a,0.0) for a in seq],dtype=np.float32)

    p=np.array([
        math.log1p(L)/8.0,
        sum(a in GROUPS['KR'] for a in seq)/L,
        sum(a in GROUPS['KRH'] for a in seq)/L,
        sum(a in GROUPS['DE'] for a in seq)/L,
        sum(a in GROUPS['POLAR'] for a in seq)/L,
        sum(a in GROUPS['HYDRO'] for a in seq)/L,
        sum(a in GROUPS['AROM'] for a in seq)/L,
        float(np.mean(q)),
    ],dtype=np.float32)

    return np.repeat(p[None,:],L,axis=0)


In [ ]:
# ============================================================
# 7. GENERATE / CACHE / ATTACH NUCLEAR-SPECIFIC BLOCKS
# ============================================================
NEW_BLOCKS = {
    'NA_SEQELECTRO32':32,
    'NA_PATCH3D40':40,
    'NA_HBOND16':16,
    'NA_TOPO12':12,
    'NA_PROTCTX8':8,
    'NA_AROMCTRL8':8,
}

if RUN_FEATURE_GENERATION:
    for split,df in SPLIT_DF.items():
        outdir=FEATURE_CACHE/split
        outdir.mkdir(parents=True,exist_ok=True)

        for n,r in enumerate(df.itertuples(),start=1):
            fp=outdir/f'{r.prot_id}.npz'

            valid=False
            if fp.exists() and not FORCE_REBUILD_NA_FEATURES:
                try:
                    z=np.load(fp)
                    valid=all(
                        z[k].shape==(len(r.sequence),d)
                        for k,d in NEW_BLOCKS.items()
                    )
                except Exception:
                    valid=False

            if not valid:
                seqb=sequence_block(r.sequence)
                patch,hbond,topo,arom=build_3d_blocks(
                    PDB_NORM_ROOT/split/f'{r.prot_id}.pdb',
                    r.sequence
                )
                prot=protein_context(r.sequence)

                np.savez_compressed(
                    fp,
                    NA_SEQELECTRO32=seqb,
                    NA_PATCH3D40=patch,
                    NA_HBOND16=hbond,
                    NA_TOPO12=topo,
                    NA_PROTCTX8=prot,
                    NA_AROMCTRL8=arom,
                )

            if n%250==0:
                print(split,n,'/',len(df))

# Attach.
for split,df in SPLIT_DF.items():
    store=ctx.stores[STORE_KEY[split]]
    outdir=FEATURE_CACHE/split

    for r in df.itertuples():
        z=np.load(outdir/f'{r.prot_id}.npz')

        for k,d in NEW_BLOCKS.items():
            a=z[k].astype(np.float32,copy=False)
            if a.shape!=(len(r.sequence),d):
                raise RuntimeError(f'{split}/{r.prot_id}/{k}: {a.shape}')
            set_block(store,r.prot_id,k,a)

print('New nuclear blocks attached.')
display(reporting.dimension_table(ctx.stores['train']))

audit_blocks = [
    'ESM','ACCESS','ELEC3D','ENV','SEQMOTIF','SPATIAL',
    *NEW_BLOCKS.keys(),
]
if EVO_AVAILABLE:
    audit_blocks.append('EVO')
if ESMIF_AVAILABLE:
    audit_blocks.append('ESMIF')

display(reporting.validate_no_direct_label_features(
    ctx.stores,
    audit_blocks
))


# Part B — Same-MLP mechanism discovery

Feature discovery is deliberately separated from architecture discovery.

Every representation in the following stages uses the same GEAR MLP and the same official five protein folds.


In [ ]:
# ============================================================
# 8. STAGE A — SINGLE-FAMILY MLP SCREEN (DEV ONLY)
# ============================================================
FAMILY_GRID = {
    'ESM':['ESM'],

    # Existing audit-supported controls
    'ESM+ACCESS':['ESM','ACCESS'],
    'ESM+ELEC3D':['ESM','ELEC3D'],
    'ESM+ENV':['ESM','ENV'],
    'ESM+SEQMOTIF':['ESM','SEQMOTIF'],
    'ESM+Structure':['ESM','SPATIAL','LOCAL'],

    # New nuclear-specific blocks
    'ESM+NA_SEQELECTRO':['ESM','NA_SEQELECTRO32'],
    'ESM+NA_PATCH3D':['ESM','NA_PATCH3D40'],
    'ESM+NA_HBOND':['ESM','NA_HBOND16'],
    'ESM+NA_TOPO':['ESM','NA_TOPO12'],
    'ESM+NA_PROTCTX':['ESM','NA_PROTCTX8'],

    # Explicit control: aromatic signal must earn its place
    'ESM+NA_AROM_CTRL':['ESM','NA_AROMCTRL8'],
}

if EVO_AVAILABLE:
    FAMILY_GRID['ESM+EVO']=['ESM','EVO']

if ESMIF_AVAILABLE:
    FAMILY_GRID['ESM+ESMIF']=['ESM','ESMIF']

if RUN_FAMILY_SCREEN:
    family_cv,family_folds=suite.run_dev_cv_only(
        ctx,
        FAMILY_GRID,
        models=('MLP',),
        run_version='nuclear_v1_01_family_mlp',
    )

    family_cv.to_csv(RUN_ROOT/'family_cv.csv',index=False)
    family_folds.to_csv(RUN_ROOT/'family_folds.csv',index=False)
else:
    family_cv=pd.read_csv(RUN_ROOT/'family_cv.csv')
    family_folds=pd.read_csv(RUN_ROOT/'family_folds.csv')

base_score=float(
    family_cv.loc[
        family_cv.representation.eq('ESM'),
        'CV_AUPRC'
    ].iloc[0]
)

family_rank=family_cv.copy()
family_rank['delta_vs_ESM']=family_rank.CV_AUPRC-base_score
family_rank=family_rank.sort_values('CV_AUPRC',ascending=False)

display(family_rank)
family_rank.to_csv(RUN_ROOT/'family_ranked.csv',index=False)

print('ESM baseline:',base_score)


In [ ]:
# ============================================================
# 9. STAGE B — AUDIT-DRIVEN MECHANISM GROUPS (DEV ONLY)
# ============================================================
MECHANISM_GRID = {
    'ESM':['ESM'],

    # Data-driven compact anchor from the audit.
    'ESM+ACCESS+ELEC3D':
        ['ESM','ACCESS','ELEC3D'],

    # Main nuclear mechanism groups.
    'ESM+BACKBONE_ATTRACTION':
        ['ESM','ACCESS','ELEC3D','NA_SEQELECTRO32'],

    'ESM+HBOND_RECOGNITION':
        ['ESM','NA_HBOND16'],

    'ESM+PATCH_ORGANIZATION':
        ['ESM','ACCESS','NA_PATCH3D40','NA_TOPO12'],

    'ESM+ELECTRO_PATCH':
        [
            'ESM',
            'ACCESS','ELEC3D',
            'NA_SEQELECTRO32',
            'NA_PATCH3D40','NA_TOPO12',
        ],

    'ESM+ELECTRO_PATCH_HBOND':
        [
            'ESM',
            'ACCESS','ELEC3D',
            'NA_SEQELECTRO32',
            'NA_PATCH3D40','NA_TOPO12',
            'NA_HBOND16',
        ],

    # Aromatic is a control, not a prior assumption.
    'ESM+ELECTRO_PATCH_HBOND+AROMCTRL':
        [
            'ESM',
            'ACCESS','ELEC3D',
            'NA_SEQELECTRO32',
            'NA_PATCH3D40','NA_TOPO12',
            'NA_HBOND16','NA_AROMCTRL8',
        ],
}

if EVO_AVAILABLE:
    MECHANISM_GRID['ESM+ELECTRO_PATCH_HBOND+EVO']=[
        'ESM',
        'ACCESS','ELEC3D',
        'NA_SEQELECTRO32',
        'NA_PATCH3D40','NA_TOPO12',
        'NA_HBOND16',
        'EVO',
    ]

if ESMIF_AVAILABLE:
    MECHANISM_GRID['ESM+ELECTRO_PATCH_HBOND+ESMIF']=[
        'ESM',
        'ACCESS','ELEC3D',
        'NA_SEQELECTRO32',
        'NA_PATCH3D40','NA_TOPO12',
        'NA_HBOND16',
        'ESMIF',
    ]

if EVO_AVAILABLE and ESMIF_AVAILABLE:
    MECHANISM_GRID['ESM+ELECTRO_PATCH_HBOND+EVO+ESMIF']=[
        'ESM',
        'ACCESS','ELEC3D',
        'NA_SEQELECTRO32',
        'NA_PATCH3D40','NA_TOPO12',
        'NA_HBOND16',
        'EVO','ESMIF',
    ]

if RUN_COMBINATION_LADDER:
    combo_cv,combo_folds=suite.run_dev_cv_only(
        ctx,
        MECHANISM_GRID,
        models=('MLP',),
        run_version='nuclear_v1_02_mechanism_combo_mlp',
    )

    combo_cv.to_csv(RUN_ROOT/'mechanism_combo_cv.csv',index=False)
    combo_folds.to_csv(RUN_ROOT/'mechanism_combo_folds.csv',index=False)
else:
    combo_cv=pd.read_csv(RUN_ROOT/'mechanism_combo_cv.csv')
    combo_folds=pd.read_csv(RUN_ROOT/'mechanism_combo_folds.csv')

combo_rank=combo_cv.sort_values('CV_AUPRC',ascending=False).reset_index(drop=True)
display(combo_rank)

BEST_REP_NAME=str(combo_rank.iloc[0].representation)
BEST_REP_BLOCKS=list(MECHANISM_GRID[BEST_REP_NAME])
BEST_REP_SCORE=float(combo_rank.iloc[0].CV_AUPRC)

print('LOCKED REPRESENTATION:',BEST_REP_NAME)
print('Blocks:',BEST_REP_BLOCKS)
print('Dev CV AUPRC:',BEST_REP_SCORE)

(RUN_ROOT/'locked_representation.json').write_text(
    json.dumps({
        'representation':BEST_REP_NAME,
        'blocks':BEST_REP_BLOCKS,
        'dev_cv_auprc':BEST_REP_SCORE,
        'selection_source':'DevSet1010 only',
    },indent=2)
)


# Part C — Multimodal architecture screen

After the feature representation is frozen, architecture is allowed to change.

Modalities are grouped semantically:

- **ESM** — PLM representation
- **ELECTRO** — ACCESS / ELEC3D / sequence electrostatics
- **PATCH** — 3D patch + patch topology
- **CHEM** — H-bond / aromatic-control chemistry
- **BIO** — EVO / motif / global protein context / learned structure when selected

Candidates:
- ConcatMLP
- ProjectedConcat
- GatedFusion
- ModalityAttention
- PatchResidualFusion
- LatentNAExpertMoE

The patch branch is residual, reflecting the Test46 observation that nuclear interfaces can be much smaller than those in Dev/Test300.


In [ ]:
# ============================================================
# 10. PACK LOCKED REPRESENTATION INTO SEMANTIC MODALITIES
# ============================================================
def modality_spec(blocks):
    blocks=list(blocks)
    spec={}

    if 'ESM' in blocks:
        spec['ESM']=['ESM']

    electro=[
        b for b in [
            'ACCESS','ELEC3D','NA_SEQELECTRO32'
        ]
        if b in blocks
    ]
    if electro:
        spec['ELECTRO']=electro

    patch=[
        b for b in [
            'NA_PATCH3D40','NA_TOPO12'
        ]
        if b in blocks
    ]
    if patch:
        spec['PATCH']=patch

    chem=[
        b for b in [
            'NA_HBOND16','NA_AROMCTRL8'
        ]
        if b in blocks
    ]
    if chem:
        spec['CHEM']=chem

    bio=[
        b for b in [
            'EVO','SEQMOTIF','NA_PROTCTX8',
            'SPATIAL','LOCAL','ESMIF'
        ]
        if b in blocks
    ]
    if bio:
        spec['BIO']=bio

    # Any selected block not yet assigned goes to OTHER.
    assigned={x for v in spec.values() for x in v}
    other=[b for b in blocks if b not in assigned]
    if other:
        spec['OTHER']=other

    return spec

MODALITY_SPEC=modality_spec(BEST_REP_BLOCKS)
print('MODALITY_SPEC =',MODALITY_SPEC)

def concat_blocks(store,pid,blocks):
    xs=[
        get_block(store,pid,b).astype(np.float32,copy=False)
        for b in blocks
    ]

    lens={x.shape[0] for x in xs}
    if len(lens)!=1:
        raise RuntimeError(f'{pid}: block length mismatch')

    return np.concatenate(xs,axis=1)

def build_pack(split):
    df=SPLIT_DF[split]
    store=ctx.stores[STORE_KEY[split]]

    mods={k:[] for k in MODALITY_SPEC}
    yy=[]
    pids=[]
    ridx=[]

    for r in df.itertuples():
        L=len(r.sequence)

        for m,blocks in MODALITY_SPEC.items():
            mods[m].append(
                concat_blocks(store,r.prot_id,blocks)
            )

        yy.append(np.asarray(r.nuclear_binding,dtype=np.float32))
        pids.extend([str(r.prot_id)]*L)
        ridx.extend(range(L))

    mods={
        k:np.concatenate(v,axis=0).astype(
            np.float16 if k in {'ESM','BIO'} else np.float32,
            copy=False
        )
        for k,v in mods.items()
    }

    return {
        'mods':mods,
        'y':np.concatenate(yy).astype(np.float32),
        'pids':np.asarray(pids,dtype=object),
        'ridx':np.asarray(ridx,dtype=np.int32),
    }

PACKS={s:build_pack(s) for s in SPLIT_DF}

DIMS={
    k:v.shape[1]
    for k,v in PACKS['DevSet1010']['mods'].items()
}

for split,p in PACKS.items():
    print(
        split,
        {k:v.shape for k,v in p['mods'].items()},
        'y=',p['y'].shape,
        'positive=',int(p['y'].sum())
    )

print('Modality dims:',DIMS)


In [ ]:
# ============================================================
# 11. MULTIMODAL MODELS
# ============================================================
class ConcatMLP(nn.Module):
    def __init__(self,dims,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)
        self.norm=nn.ModuleDict({
            k:nn.LayerNorm(d)
            for k,d in dims.items()
        })
        d=sum(dims.values())

        self.head=nn.Sequential(
            nn.Linear(d,512),nn.GELU(),nn.Dropout(drop),
            nn.Linear(512,128),nn.GELU(),nn.Dropout(drop),
            nn.Linear(128,1),
        )

    def forward(self,x):
        z=torch.cat(
            [self.norm[k](x[k]) for k in self.keys],
            dim=-1
        )
        return self.head(z).squeeze(-1)

class ProjectedConcat(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)
        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.head=nn.Sequential(
            nn.Linear(latent*len(dims),256),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(256,1)
        )

    def forward(self,x):
        z=torch.cat(
            [self.enc[k](x[k]) for k in self.keys],
            dim=-1
        )
        return self.head(z).squeeze(-1)

class GatedFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in dims
        })

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def encode(self,x):
        zz=[]
        gg=[]

        for k in self.keys:
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.gate[k](z))

        Z=torch.stack(zz,dim=1)
        G=torch.softmax(torch.cat(gg,dim=1),dim=1).unsqueeze(-1)

        return (Z*G).sum(dim=1),G.squeeze(-1)

    def forward(self,x):
        h,_=self.encode(x)
        return self.head(h).squeeze(-1)

class ModalityAttention(nn.Module):
    def __init__(self,dims,latent=192,heads=6,drop=0.2):
        super().__init__()
        assert latent%heads==0

        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.attn=nn.MultiheadAttention(
            latent,
            heads,
            dropout=drop,
            batch_first=True
        )

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def forward(self,x):
        Z=torch.stack(
            [self.enc[k](x[k]) for k in self.keys],
            dim=1
        )
        A,_=self.attn(Z,Z,Z,need_weights=False)
        h=(Z+A).mean(dim=1)
        return self.head(h).squeeze(-1)

class PatchResidualFusion(nn.Module):
    def __init__(self,dims,latent=192,drop=0.2):
        super().__init__()

        if 'PATCH' not in dims:
            raise ValueError('PatchResidualFusion requires PATCH modality')

        self.base_keys=sorted(
            [k for k in dims if k!='PATCH']
        )

        self.base_enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
            if k!='PATCH'
        })

        self.patch_enc=nn.Sequential(
            nn.LayerNorm(dims['PATCH']),
            nn.Linear(dims['PATCH'],latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent,latent),
        )

        self.base_gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in self.base_keys
        })

        # sigmoid(-2.944) ≈ 0.05
        self.alpha_logit=nn.Parameter(
            torch.tensor(-2.944,dtype=torch.float32)
        )

        self.head=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,192),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(192,1)
        )

    def forward(self,x):
        zz=[]
        gg=[]

        for k in self.base_keys:
            z=self.base_enc[k](x[k])
            zz.append(z)
            gg.append(self.base_gate[k](z))

        Z=torch.stack(zz,dim=1)
        G=torch.softmax(torch.cat(gg,dim=1),dim=1).unsqueeze(-1)
        h=(Z*G).sum(dim=1)

        patch=self.patch_enc(x['PATCH'])
        alpha=torch.sigmoid(self.alpha_logit)

        h=h+alpha*patch
        return self.head(h).squeeze(-1)

class LatentNAExpertMoE(nn.Module):
    def __init__(self,dims,latent=192,experts=4,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        self.mod_gate=nn.ModuleDict({
            k:nn.Linear(latent,1)
            for k in dims
        })

        self.shared=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,latent),
            nn.GELU()
        )

        self.experts=nn.ModuleList([
            nn.Sequential(
                nn.Linear(latent,128),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(128,1)
            )
            for _ in range(experts)
        ])

        self.expert_gate=nn.Linear(latent,experts)

    def forward(self,x):
        zz=[]
        gg=[]

        for k in self.keys:
            z=self.enc[k](x[k])
            zz.append(z)
            gg.append(self.mod_gate[k](z))

        Z=torch.stack(zz,dim=1)
        G=torch.softmax(torch.cat(gg,dim=1),dim=1).unsqueeze(-1)

        h=self.shared((Z*G).sum(dim=1))

        logits=torch.cat(
            [e(h) for e in self.experts],
            dim=1
        )
        weights=torch.softmax(
            self.expert_gate(h),
            dim=1
        )

        return (logits*weights).sum(dim=1)

ARCH_FACTORY = {
    'ConcatMLP':lambda dims:ConcatMLP(dims),
    'ProjectedConcat':lambda dims:ProjectedConcat(dims),
    'GatedFusion':lambda dims:GatedFusion(dims),
    'ModalityAttention':lambda dims:ModalityAttention(dims),
    'LatentNAExpertMoE':lambda dims:LatentNAExpertMoE(dims),
}

if 'PATCH' in DIMS:
    ARCH_FACTORY['PatchResidualFusion']=lambda dims:PatchResidualFusion(dims)

print('Architectures:',list(ARCH_FACTORY))


In [ ]:
# ============================================================
# 12. CUSTOM CV TRAINER — OFFICIAL PROTEIN FOLDS, RESUME SAFE
# ============================================================
class MultiArrayDataset(Dataset):
    def __init__(self,mods,y):
        self.mods=mods
        self.y=y
        self.keys=sorted(mods)

    def __len__(self):
        return len(self.y)

    def __getitem__(self,i):
        x={
            k:torch.from_numpy(
                np.asarray(self.mods[k][i],dtype=np.float32)
            )
            for k in self.keys
        }
        return x,torch.tensor(self.y[i],dtype=torch.float32)

def collate_multi(batch):
    keys=sorted(batch[0][0])
    x={
        k:torch.stack([b[0][k] for b in batch],dim=0)
        for k in keys
    }
    y=torch.stack([b[1] for b in batch])
    return x,y

def ap(y,p):
    return float(average_precision_score(y,p))

def predict_flat(model,dataset,indices=None,batch_size=8192):
    ds=dataset if indices is None else Subset(
        dataset,
        indices.tolist() if isinstance(indices,np.ndarray) else indices
    )

    loader=DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_multi
    )

    out=[]
    model.eval()

    with torch.no_grad():
        for x,_ in loader:
            x={
                k:v.to(DEVICE,non_blocking=True)
                for k,v in x.items()
            }
            out.append(
                torch.sigmoid(model(x)).detach().cpu().numpy()
            )

    return np.concatenate(out)

CUSTOM_STATE_ROOT=RUN_ROOT/'custom_states'
CUSTOM_STATE_ROOT.mkdir(parents=True,exist_ok=True)

def train_one_arch_fold(arch,pack,train_idx,val_idx,fold):
    torch.manual_seed(SEED+fold)
    np.random.seed(SEED+fold)

    ds=MultiArrayDataset(pack['mods'],pack['y'])

    train_ds=Subset(ds,train_idx.tolist())
    val_ds=Subset(ds,val_idx.tolist())

    train_loader=DataLoader(
        train_ds,
        batch_size=CUSTOM_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_multi
    )

    model=ARCH_FACTORY[arch](DIMS).to(DEVICE)

    ytr=pack['y'][train_idx]
    npos=max(float(ytr.sum()),1.0)
    nneg=max(float(len(ytr)-ytr.sum()),1.0)

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            np.sqrt(nneg/npos),
            dtype=torch.float32,
            device=DEVICE
        )
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=CUSTOM_LR,
        weight_decay=CUSTOM_WEIGHT_DECAY
    )

    best_ap=-1.0
    best_state=None
    bad=0

    for epoch in range(1,CUSTOM_EPOCHS+1):
        model.train()
        losses=[]

        for x,y in train_loader:
            x={k:v.to(DEVICE,non_blocking=True) for k,v in x.items()}
            y=y.to(DEVICE,non_blocking=True)

            opt.zero_grad(set_to_none=True)

            logits=model(x)
            loss=criterion(logits,y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            opt.step()

            losses.append(float(loss.detach().cpu()))

        vp=predict_flat(model,ds,val_idx)
        vy=pack['y'][val_idx]
        val_ap=ap(vy,vp)

        if val_ap>best_ap+1e-5:
            best_ap=val_ap
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            bad=0
        else:
            bad+=1

        if epoch==1 or epoch%5==0:
            print(
                arch,'fold',fold,'epoch',epoch,
                'loss',round(float(np.mean(losses)),4),
                'val_AP',round(val_ap,5),
                'best',round(best_ap,5)
            )

        if bad>=CUSTOM_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model,best_state

def run_arch_cv(arch):
    pack=PACKS['DevSet1010']
    pids=pack['pids']
    y=pack['y']

    ds=MultiArrayDataset(pack['mods'],pack['y'])
    oof=np.full(len(y),np.nan,dtype=np.float32)

    rows=[]
    states={}

    adir=CUSTOM_STATE_ROOT/arch
    adir.mkdir(parents=True,exist_ok=True)

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(val_ids)

        val_idx=np.flatnonzero(
            np.array([p in val_set for p in pids])
        )
        train_idx=np.flatnonzero(
            np.array([p not in val_set for p in pids])
        )

        state_fp=adir/f'fold{fold}.pt'
        pred_fp=adir/f'fold{fold}_val.npy'
        meta_fp=adir/f'fold{fold}.json'

        reuse=False

        if state_fp.exists() and pred_fp.exists() and meta_fp.exists():
            try:
                meta=json.loads(meta_fp.read_text())
                pp=np.load(pred_fp)

                reuse=(
                    meta.get('representation')==BEST_REP_NAME
                    and meta.get('n_val')==len(val_idx)
                    and pp.shape==(len(val_idx),)
                    and np.all(np.isfinite(pp))
                )
            except Exception:
                reuse=False

        if reuse:
            print(arch,'fold',fold,'REUSE')
            state=torch.load(state_fp,map_location='cpu')
            pred=np.load(pred_fp).astype(np.float32)
        else:
            print(arch,'fold',fold,'TRAIN')
            model,state=train_one_arch_fold(
                arch,pack,train_idx,val_idx,fold
            )
            pred=predict_flat(model,ds,val_idx).astype(np.float32)

            torch.save(state,state_fp)
            np.save(pred_fp,pred)
            meta_fp.write_text(json.dumps({
                'representation':BEST_REP_NAME,
                'blocks':BEST_REP_BLOCKS,
                'n_val':int(len(val_idx)),
            },indent=2))

            del model
            torch.cuda.empty_cache()

        states[fold]=state
        oof[val_idx]=pred

        rows.append({
            'architecture':arch,
            'fold':fold,
            'AUPRC':ap(y[val_idx],pred),
            'n_val':len(val_idx),
            'n_positive':int(y[val_idx].sum()),
        })

    if not np.all(np.isfinite(oof)):
        raise RuntimeError(f'{arch}: incomplete OOF')

    f=pd.DataFrame(rows)

    return (
        {
            'architecture':arch,
            'CV_AUPRC_mean':float(f.AUPRC.mean()),
            'CV_AUPRC_std':float(f.AUPRC.std(ddof=1)),
            'OOF_AUPRC':ap(y,oof),
        },
        f,
        oof,
        states
    )


In [ ]:
# ============================================================
# 13. RUN MULTIMODAL ARCHITECTURE SCREEN — DEV ONLY
# ============================================================
ARCH_RESULTS={}
ARCH_FOLDS=[]
ARCH_OOF={}
ARCH_STATES={}

if RUN_CUSTOM_ARCHITECTURES:
    for arch in ARCH_FACTORY:
        print('\n'+'='*80)
        print('ARCH:',arch)
        print('='*80)

        s,f,o,st=run_arch_cv(arch)

        ARCH_RESULTS[arch]=s
        ARCH_FOLDS.append(f)
        ARCH_OOF[arch]=o
        ARCH_STATES[arch]=st

arch_summary=pd.DataFrame(
    ARCH_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(drop=True)

arch_folds=pd.concat(
    ARCH_FOLDS,
    ignore_index=True
)

arch_summary.to_csv(RUN_ROOT/'architecture_cv.csv',index=False)
arch_folds.to_csv(RUN_ROOT/'architecture_folds.csv',index=False)

display(arch_summary)

concat_score=float(
    arch_summary.loc[
        arch_summary.architecture.eq('ConcatMLP'),
        'CV_AUPRC_mean'
    ].iloc[0]
)

best_arch=str(arch_summary.iloc[0].architecture)
best_arch_score=float(arch_summary.iloc[0].CV_AUPRC_mean)

if best_arch_score-concat_score >= MIN_MEAN_AUPRC_GAIN:
    LOCKED_ARCH=best_arch
else:
    LOCKED_ARCH='ConcatMLP'

LOCKED_ARCH_SCORE=float(
    arch_summary.loc[
        arch_summary.architecture.eq(LOCKED_ARCH),
        'CV_AUPRC_mean'
    ].iloc[0]
)

print('LOCKED ARCH:',LOCKED_ARCH,LOCKED_ARCH_SCORE)

(RUN_ROOT/'locked_architecture.json').write_text(
    json.dumps({
        'architecture':LOCKED_ARCH,
        'dev_cv_auprc':LOCKED_ARCH_SCORE,
        'concat_cv_auprc':concat_score,
        'best_raw_architecture':best_arch,
        'best_raw_cv_auprc':best_arch_score,
    },indent=2)
)


# Part D — Nuclear surface-patch hypergraph

Generic residue graphs already underperformed in the previous nuclear experiments.

V1 therefore tests a task-specific graph unit:

```text
residues
   ↓
candidate cationic/polar 3D surface patches
   ↓
patch representation
   ↓
patch → residue residual message
```

Candidate patches are label-independent and seeded by K/R/H/N/Q/S/T/Y residues. All nearby residues within 10 Å become patch members.

The hypergraph correction starts with a learned residual weight near 0.05, so the patch branch cannot dominate the strong local predictor at initialization.


In [ ]:
# ============================================================
# 14. BUILD / CACHE NUCLEAR CANDIDATE PATCH HYPERGRAPHS
# ============================================================
PATCH_CACHE=CACHE_ROOT/'patch_hypergraph_v1'
PATCH_CACHE.mkdir(parents=True,exist_ok=True)

PATCH_SEEDS=set('KRHNQSTY')

def structure_coords(split,pid,seq):
    path=PDB_NORM_ROOT/split/f'{pid}.pdb'
    st=PDBParser(QUIET=True).get_structure('p',str(path))
    m=next(st.get_models())
    rr=standard_residues(m['A'])

    if residue_sequence(rr)!=seq:
        raise RuntimeError(f'{split}/{pid}: normalized PDB mismatch')

    return np.stack(
        [representative_coord(r) for r in rr],
        axis=0
    ).astype(np.float32)

def patch_static_features(seq,coords,members):
    members=np.asarray(members,dtype=int)
    aa=[seq[i] for i in members]
    cc=coords[members]
    n=len(members)

    q=np.asarray(
        [CHARGE.get(x,0.0) for x in aa],
        dtype=np.float32
    )

    centroid=cc.mean(axis=0)
    radial=np.linalg.norm(cc-centroid[None,:],axis=1)

    pair=[]
    if n>=2:
        for i in range(n):
            for j in range(i+1,n):
                pair.append(float(np.linalg.norm(cc[i]-cc[j])))

    pair=np.asarray(pair if pair else [0.0],dtype=np.float32)

    probs=[]
    cnt=Counter(aa)
    for x in cnt.values():
        probs.append(x/n)

    entropy=(
        -sum(p*math.log(p+1e-12) for p in probs)
        / math.log(20)
    )

    f=np.zeros(18,dtype=np.float32)

    f[0]=min(n,40)/40.0
    f[1]=sum(x in GROUPS['KR'] for x in aa)/n
    f[2]=sum(x in GROUPS['KRH'] for x in aa)/n
    f[3]=sum(x in GROUPS['DE'] for x in aa)/n
    f[4]=sum(x in GROUPS['POLAR'] for x in aa)/n
    f[5]=sum(x in GROUPS['AROM'] for x in aa)/n
    f[6]=sum(x in GROUPS['HYDRO'] for x in aa)/n
    f[7]=float(np.mean(q))
    f[8]=sum(x in GROUPS['HBD'] for x in aa)/n
    f[9]=sum(x in GROUPS['HBA'] for x in aa)/n

    f[10]=min(float(radial.mean())/12.0,1.0)
    f[11]=min(float(radial.std())/8.0,1.0)
    f[12]=min(float(radial.max())/15.0,1.0)

    f[13]=min((members.max()-members.min())/100.0,1.0)
    f[14]=min(float(pair.mean())/15.0,1.0)
    f[15]=min(float(pair.std())/10.0,1.0)
    f[16]=float(entropy)
    f[17]=float(np.tanh(
        (
            sum(x in GROUPS['KRH'] for x in aa)
            - sum(x in GROUPS['DE'] for x in aa)
        )/5.0
    ))

    return f

def build_patch_graph(split,pid,seq):
    coords=structure_coords(split,pid,seq)
    tree=cKDTree(coords)

    candidates={}

    for seed,a in enumerate(seq):
        if a not in PATCH_SEEDS:
            continue

        members=tuple(sorted(
            tree.query_ball_point(coords[seed],PATCH_RADIUS_A)
        ))

        if len(members)<4:
            continue

        if not any(seq[j] in GROUPS['KRH'] for j in members):
            continue

        f=patch_static_features(seq,coords,members)

        # Label-independent ranking score for cationic/polar recognition plausibility.
        score=(
            2.0*f[2]
            + 0.7*f[4]
            - 0.7*f[3]
            + 0.2*f[0]
        )

        old=candidates.get(members)
        if old is None or score>old[0]:
            candidates[members]=(score,f)

    ranked=sorted(
        candidates.items(),
        key=lambda kv:kv[1][0],
        reverse=True
    )[:PATCH_MAX_PER_PROTEIN]

    edge_res=[]
    edge_patch=[]
    patch_feat=[]

    for pidx,(members,(_,feat)) in enumerate(ranked):
        patch_feat.append(feat)

        for r in members:
            edge_res.append(int(r))
            edge_patch.append(int(pidx))

    return (
        np.asarray(edge_res,dtype=np.int64),
        np.asarray(edge_patch,dtype=np.int64),
        np.asarray(patch_feat,dtype=np.float32).reshape(-1,18)
    )

PATCH_RAM={}
PATCH_AUDIT=[]

for split,df in SPLIT_DF.items():
    outdir=PATCH_CACHE/split
    outdir.mkdir(parents=True,exist_ok=True)

    for n,r in enumerate(df.itertuples(),start=1):
        fp=outdir/f'{r.prot_id}.npz'

        valid=False

        if fp.exists() and not FORCE_REBUILD_PATCHES:
            try:
                z=np.load(fp)
                valid=(
                    z['patch_feat'].ndim==2
                    and z['patch_feat'].shape[1]==18
                    and len(z['edge_res'])==len(z['edge_patch'])
                )
            except Exception:
                valid=False

        if not valid:
            er,ep,pf=build_patch_graph(
                split,
                str(r.prot_id),
                r.sequence
            )

            np.savez_compressed(
                fp,
                edge_res=er,
                edge_patch=ep,
                patch_feat=pf
            )

        z=np.load(fp)

        er=z['edge_res'].astype(np.int64)
        ep=z['edge_patch'].astype(np.int64)
        pf=z['patch_feat'].astype(np.float32)

        PATCH_RAM[(split,str(r.prot_id))]=(er,ep,pf)

        covered=len(np.unique(er)) if len(er) else 0

        PATCH_AUDIT.append({
            'dataset':split,
            'prot_id':str(r.prot_id),
            'n_residues':len(r.sequence),
            'n_candidate_patches':len(pf),
            'n_memberships':len(er),
            'covered_residue_fraction':
                covered/max(len(r.sequence),1),
        })

        if n%250==0:
            print(split,n,'/',len(df))

patch_audit=pd.DataFrame(PATCH_AUDIT)
patch_audit.to_csv(RUN_ROOT/'patch_audit.csv',index=False)

display(
    patch_audit.groupby('dataset').agg(
        proteins=('prot_id','nunique'),
        mean_patches=('n_candidate_patches','mean'),
        median_patches=('n_candidate_patches','median'),
        mean_memberships=('n_memberships','mean'),
        mean_covered_fraction=('covered_residue_fraction','mean'),
    ).reset_index()
)

print('Patch hypergraph cache: PASS')


In [ ]:
# ============================================================
# 15. PATCH-HYPERGRAPH DATASET / MODEL
# ============================================================
class PatchProteinDataset(Dataset):
    def __init__(self,split,df):
        self.split=split
        self.df=df.reset_index(drop=True)
        self.store=ctx.stores[STORE_KEY[split]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self,i):
        r=self.df.iloc[i]
        pid=str(r.prot_id)

        mods={
            m:concat_blocks(
                self.store,
                pid,
                blocks
            ).astype(np.float32,copy=False)
            for m,blocks in MODALITY_SPEC.items()
        }

        er,ep,pf=PATCH_RAM[(self.split,pid)]

        return {
            'pid':pid,
            'mods':mods,
            'y':np.asarray(r.nuclear_binding,dtype=np.float32),
            'edge_res':er,
            'edge_patch':ep,
            'patch_feat':pf,
        }

def collate_patch(batch):
    keys=sorted(batch[0]['mods'])

    mods={
        k:torch.from_numpy(
            np.concatenate([b['mods'][k] for b in batch],axis=0)
        ).float()
        for k in keys
    }

    y=torch.from_numpy(
        np.concatenate([b['y'] for b in batch])
    ).float()

    patch_feat=[]
    edge_res=[]
    edge_patch=[]

    roff=0
    poff=0

    for b in batch:
        L=len(b['y'])
        P=len(b['patch_feat'])

        if P:
            patch_feat.append(
                torch.from_numpy(b['patch_feat']).float()
            )

        if len(b['edge_res']):
            edge_res.append(
                torch.from_numpy(b['edge_res']).long()+roff
            )
            edge_patch.append(
                torch.from_numpy(b['edge_patch']).long()+poff
            )

        roff+=L
        poff+=P

    patch_feat=(
        torch.cat(patch_feat,dim=0)
        if patch_feat
        else torch.zeros((0,18),dtype=torch.float32)
    )

    edge_res=(
        torch.cat(edge_res,dim=0)
        if edge_res
        else torch.zeros((0,),dtype=torch.long)
    )

    edge_patch=(
        torch.cat(edge_patch,dim=0)
        if edge_patch
        else torch.zeros((0,),dtype=torch.long)
    )

    return {
        'mods':mods,
        'y':y,
        'edge_res':edge_res,
        'edge_patch':edge_patch,
        'patch_feat':patch_feat,
    }

class PatchResidueEncoder(nn.Module):
    def __init__(self,dims,mode='gated',latent=192,drop=0.2):
        super().__init__()
        self.keys=sorted(dims)
        self.mode=mode

        self.enc=nn.ModuleDict({
            k:nn.Sequential(
                nn.LayerNorm(d),
                nn.Linear(d,latent),
                nn.GELU(),
                nn.Dropout(drop)
            )
            for k,d in dims.items()
        })

        if mode=='gated':
            self.gate=nn.ModuleDict({
                k:nn.Linear(latent,1)
                for k in dims
            })
        elif mode=='attention':
            self.attn=nn.MultiheadAttention(
                latent,
                6,
                dropout=drop,
                batch_first=True
            )
        else:
            raise ValueError(mode)

    def forward(self,x):
        Z=torch.stack(
            [self.enc[k](x[k]) for k in self.keys],
            dim=1
        )

        if self.mode=='gated':
            G=torch.cat(
                [
                    self.gate[k](Z[:,j,:])
                    for j,k in enumerate(self.keys)
                ],
                dim=1
            )
            G=torch.softmax(G,dim=1).unsqueeze(-1)
            return (Z*G).sum(dim=1)

        A,_=self.attn(Z,Z,Z,need_weights=False)
        return (Z+A).mean(dim=1)

class NuclearPatchHypergraph(nn.Module):
    def __init__(
        self,
        dims,
        mode='gated',
        latent=192,
        moe=False,
        experts=4,
        drop=0.2,
    ):
        super().__init__()

        self.encoder=PatchResidueEncoder(
            dims,
            mode=mode,
            latent=latent,
            drop=drop
        )

        self.patch_static=nn.Sequential(
            nn.LayerNorm(18),
            nn.Linear(18,64),
            nn.GELU()
        )

        self.patch_mlp=nn.Sequential(
            nn.Linear(latent*2+64,latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent,latent),
            nn.GELU(),
        )

        self.message_mlp=nn.Sequential(
            nn.LayerNorm(latent),
            nn.Linear(latent,latent),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(latent,latent)
        )

        self.alpha_logit=nn.Parameter(
            torch.tensor(-2.944,dtype=torch.float32)
        )

        self.moe=moe

        if not moe:
            self.head=nn.Sequential(
                nn.LayerNorm(latent),
                nn.Linear(latent,128),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(128,1)
            )
        else:
            self.experts=nn.ModuleList([
                nn.Sequential(
                    nn.LayerNorm(latent),
                    nn.Linear(latent,128),
                    nn.GELU(),
                    nn.Dropout(drop),
                    nn.Linear(128,1)
                )
                for _ in range(experts)
            ])
            self.expert_gate=nn.Linear(latent,experts)

    def index_mean(self,x,index,n):
        out=x.new_zeros((n,x.shape[-1]))
        cnt=x.new_zeros((n,1))

        if len(index):
            out.index_add_(0,index,x)
            cnt.index_add_(
                0,index,
                x.new_ones((len(index),1))
            )

        return out/cnt.clamp_min(1.0)

    def index_max(self, x, index, n):
        if n == 0:
            return x.new_zeros(
                (0, x.shape[-1])
            )

        base = x.new_full(
            (n, x.shape[-1]),
            -1e9
        )

        if len(index):
            idx = index[:, None].expand(
                -1,
                x.shape[-1]
            )

            # IMPORTANT:
            # out-of-place scatter_reduce for autograd safety
            out = base.scatter_reduce(
                0,
                idx,
                x,
                reduce='amax',
                include_self=True
            )

        else:
            out = base

        # IMPORTANT:
        # no inplace modification after scatter_reduce
        out = torch.where(
            out < -1e8,
            torch.zeros_like(out),
            out
        )

        return out
    def forward(self,mods,edge_res,edge_patch,patch_feat):
        h=self.encoder(mods)

        nres=h.shape[0]
        npatch=patch_feat.shape[0]

        alpha=torch.sigmoid(self.alpha_logit)

        if npatch and edge_res.numel():
            mean=self.index_mean(
                h[edge_res],
                edge_patch,
                npatch
            )

            mx=self.index_max(
                h[edge_res],
                edge_patch,
                npatch
            )

            ps=self.patch_static(patch_feat)

            hp=self.patch_mlp(
                torch.cat([mean,mx,ps],dim=-1)
            )

            msg=self.index_mean(
                hp[edge_patch],
                edge_res,
                nres
            )

            incident=h.new_zeros((nres,1))
            incident.index_add_(
                0,
                edge_res,
                h.new_ones((len(edge_res),1))
            )

            delta=self.message_mlp(msg)
            delta=delta*(incident>0).float()

            h=h+alpha*delta

        if not self.moe:
            return self.head(h).squeeze(-1)

        logits=torch.cat(
            [e(h) for e in self.experts],
            dim=1
        )

        w=torch.softmax(
            self.expert_gate(h),
            dim=1
        )

        return (logits*w).sum(dim=1)

PATCH_FACTORY={
    'NA_PatchHG_Gated':lambda dims:NuclearPatchHypergraph(
        dims,mode='gated',latent=PATCH_LATENT,moe=False
    ),
    'NA_PatchHG_Attention':lambda dims:NuclearPatchHypergraph(
        dims,mode='attention',latent=PATCH_LATENT,moe=False
    ),
    'NA_PatchHG_GatedMoE':lambda dims:NuclearPatchHypergraph(
        dims,mode='gated',latent=PATCH_LATENT,moe=True,experts=4
    ),
}

print('Patch-hypergraph architectures:',list(PATCH_FACTORY))


In [ ]:
# ============================================================
# 16. PATCH-HYPERGRAPH CV TRAINER — RESUME SAFE
# ============================================================
PATCH_STATE_ROOT=RUN_ROOT/'patch_states'
PATCH_STATE_ROOT.mkdir(parents=True,exist_ok=True)

def move_patch_batch(b):
    return {
        'mods':{
            k:v.to(DEVICE,non_blocking=True)
            for k,v in b['mods'].items()
        },
        'y':b['y'].to(DEVICE,non_blocking=True),
        'edge_res':b['edge_res'].to(DEVICE,non_blocking=True),
        'edge_patch':b['edge_patch'].to(DEVICE,non_blocking=True),
        'patch_feat':b['patch_feat'].to(DEVICE,non_blocking=True),
    }

def predict_patch(model,dataset):
    loader=DataLoader(
        dataset,
        batch_size=PATCH_PROTEIN_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_patch
    )

    pp=[]
    yy=[]

    model.eval()

    with torch.no_grad():
        for b in loader:
            b=move_patch_batch(b)

            p=torch.sigmoid(
                model(
                    b['mods'],
                    b['edge_res'],
                    b['edge_patch'],
                    b['patch_feat']
                )
            )

            pp.append(p.detach().cpu().numpy())
            yy.append(b['y'].detach().cpu().numpy())

    return np.concatenate(pp),np.concatenate(yy)

def train_patch_fold(arch,tr_ds,va_ds,fold):
    torch.manual_seed(SEED+100+fold)
    np.random.seed(SEED+100+fold)

    model=PATCH_FACTORY[arch](DIMS).to(DEVICE)

    ytr=np.concatenate(
        tr_ds.df.nuclear_binding.apply(
            lambda x:np.asarray(x,dtype=np.float32)
        ).tolist()
    )

    npos=max(float(ytr.sum()),1.0)
    nneg=max(float(len(ytr)-ytr.sum()),1.0)

    criterion=nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            np.sqrt(nneg/npos),
            dtype=torch.float32,
            device=DEVICE
        )
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=PATCH_LR,
        weight_decay=PATCH_WEIGHT_DECAY
    )

    loader=DataLoader(
        tr_ds,
        batch_size=PATCH_PROTEIN_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_patch
    )

    best_ap=-1.0
    best_state=None
    bad=0

    for epoch in range(1,PATCH_EPOCHS+1):
        model.train()
        losses=[]

        for b in loader:
            b=move_patch_batch(b)

            opt.zero_grad(set_to_none=True)

            logits=model(
                b['mods'],
                b['edge_res'],
                b['edge_patch'],
                b['patch_feat']
            )

            loss=criterion(logits,b['y'])
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            opt.step()

            losses.append(float(loss.detach().cpu()))

        vp,vy=predict_patch(model,va_ds)
        val_ap=ap(vy,vp)

        if val_ap>best_ap+1e-5:
            best_ap=val_ap
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            bad=0
        else:
            bad+=1

        if epoch==1 or epoch%5==0:
            print(
                arch,'fold',fold,'epoch',epoch,
                'loss',round(float(np.mean(losses)),4),
                'val_AP',round(val_ap,5),
                'best',round(best_ap,5),
                'alpha',round(
                    float(torch.sigmoid(model.alpha_logit).detach().cpu()),
                    4
                )
            )

        if bad>=PATCH_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model,best_state

def run_patch_cv(arch):
    offsets={}
    cursor=0

    for r in train_df.itertuples():
        L=len(r.sequence)
        offsets[str(r.prot_id)]=(cursor,cursor+L)
        cursor+=L

    oof=np.full(cursor,np.nan,dtype=np.float32)

    rows=[]
    states={}

    adir=PATCH_STATE_ROOT/arch
    adir.mkdir(parents=True,exist_ok=True)

    for fold,val_ids in OFFICIAL_FOLDS.items():
        val_set=set(val_ids)

        tr_df=train_df[
            ~train_df.prot_id.isin(val_set)
        ].copy()

        va_df=train_df[
            train_df.prot_id.isin(val_set)
        ].copy()

        tr_ds=PatchProteinDataset('DevSet1010',tr_df)
        va_ds=PatchProteinDataset('DevSet1010',va_df)

        state_fp=adir/f'fold{fold}.pt'
        pred_fp=adir/f'fold{fold}_val.npy'
        meta_fp=adir/f'fold{fold}.json'

        expected=int(va_df.sequence.str.len().sum())

        reuse=False

        if state_fp.exists() and pred_fp.exists() and meta_fp.exists():
            try:
                meta=json.loads(meta_fp.read_text())
                pp=np.load(pred_fp)

                reuse=(
                    meta.get('representation')==BEST_REP_NAME
                    and meta.get('n_val')==expected
                    and pp.shape==(expected,)
                    and np.all(np.isfinite(pp))
                )
            except Exception:
                reuse=False

        if reuse:
            print(arch,'fold',fold,'REUSE')
            state=torch.load(state_fp,map_location='cpu')
            pred=np.load(pred_fp).astype(np.float32)
        else:
            print(arch,'fold',fold,'TRAIN')
            model,state=train_patch_fold(
                arch,tr_ds,va_ds,fold
            )
            pred,_=predict_patch(model,va_ds)

            torch.save(state,state_fp)
            np.save(pred_fp,pred)

            meta_fp.write_text(json.dumps({
                'representation':BEST_REP_NAME,
                'blocks':BEST_REP_BLOCKS,
                'n_val':expected,
            },indent=2))

            del model
            torch.cuda.empty_cache()

        states[fold]=state

        pos=0

        for r in va_df.itertuples():
            L=len(r.sequence)
            a,b=offsets[str(r.prot_id)]
            oof[a:b]=pred[pos:pos+L]
            pos+=L

        yv=np.concatenate(
            va_df.nuclear_binding.apply(
                lambda x:np.asarray(x,dtype=np.float32)
            ).tolist()
        )

        rows.append({
            'architecture':arch,
            'fold':fold,
            'AUPRC':ap(yv,pred),
            'n_val':len(pred),
            'n_positive':int(yv.sum()),
        })

    if not np.all(np.isfinite(oof)):
        raise RuntimeError(f'{arch}: incomplete patch OOF')

    f=pd.DataFrame(rows)
    yall=np.concatenate(
        train_df.nuclear_binding.apply(
            lambda x:np.asarray(x,dtype=np.float32)
        ).tolist()
    )

    return (
        {
            'architecture':arch,
            'CV_AUPRC_mean':float(f.AUPRC.mean()),
            'CV_AUPRC_std':float(f.AUPRC.std(ddof=1)),
            'OOF_AUPRC':ap(yall,oof),
        },
        f,
        oof,
        states
    )


In [ ]:
# ============================================================
# 17. RUN PATCH-HYPERGRAPH SCREEN — DEV ONLY
# ============================================================
PATCH_RESULTS={}
PATCH_FOLDS=[]
PATCH_OOF={}
PATCH_STATES={}

if RUN_PATCH_HYPERGRAPH:
    for arch in PATCH_FACTORY:
        print('\n'+'='*80)
        print('PATCH HYPERGRAPH:',arch)
        print('='*80)

        s,f,o,st=run_patch_cv(arch)

        PATCH_RESULTS[arch]=s
        PATCH_FOLDS.append(f)
        PATCH_OOF[arch]=o
        PATCH_STATES[arch]=st

patch_summary=pd.DataFrame(
    PATCH_RESULTS.values()
).sort_values(
    'CV_AUPRC_mean',
    ascending=False
).reset_index(drop=True)

patch_folds=pd.concat(
    PATCH_FOLDS,
    ignore_index=True
)

patch_summary.to_csv(RUN_ROOT/'patch_hypergraph_cv.csv',index=False)
patch_folds.to_csv(RUN_ROOT/'patch_hypergraph_folds.csv',index=False)

display(patch_summary)

BEST_PATCH=str(patch_summary.iloc[0].architecture)
BEST_PATCH_SCORE=float(patch_summary.iloc[0].CV_AUPRC_mean)

alphas=[]
for fold,state in PATCH_STATES[BEST_PATCH].items():
    if 'alpha_logit' in state:
        alphas.append(
            float(torch.sigmoid(state['alpha_logit']).item())
        )

print('BEST PATCH MODEL:',BEST_PATCH)
print('Dev CV AUPRC:',BEST_PATCH_SCORE)
print('Residual alpha:',alphas)
print('Mean alpha:',float(np.mean(alphas)) if alphas else None)

(RUN_ROOT/'locked_patch_model.json').write_text(
    json.dumps({
        'architecture':BEST_PATCH,
        'dev_cv_auprc':BEST_PATCH_SCORE,
        'alpha_by_fold':alphas,
        'alpha_mean':float(np.mean(alphas)) if alphas else None,
    },indent=2)
)


# Part E — Global Dev-only lock

The final headline family is chosen **before** external evaluation.

A more complex patch-hypergraph model replaces the best local multimodal model only if it improves mean Dev CV AUPRC by at least the configured complexity margin.


In [ ]:
# ============================================================
# 18. GLOBAL DEV-ONLY LOCK
# ============================================================
candidates=pd.DataFrame([
    {
        'family':'LOCAL_MULTIMODAL',
        'model':LOCKED_ARCH,
        'Dev_CV_AUPRC':LOCKED_ARCH_SCORE,
        'complexity_rank':1,
    },
    {
        'family':'PATCH_HYPERGRAPH',
        'model':BEST_PATCH,
        'Dev_CV_AUPRC':BEST_PATCH_SCORE,
        'complexity_rank':2,
    },
])

winner=candidates.sort_values('complexity_rank').iloc[0].to_dict()

for _,r in candidates.sort_values('complexity_rank').iloc[1:].iterrows():
    if (
        float(r.Dev_CV_AUPRC)
        - float(winner['Dev_CV_AUPRC'])
        >= GLOBAL_COMPLEXITY_MARGIN
    ):
        winner=r.to_dict()

GLOBAL_FAMILY=str(winner['family'])
GLOBAL_MODEL=str(winner['model'])
GLOBAL_SCORE=float(winner['Dev_CV_AUPRC'])

display(candidates)

print('GLOBAL LOCK:',GLOBAL_FAMILY,GLOBAL_MODEL,GLOBAL_SCORE)

(RUN_ROOT/'GLOBAL_LOCK.json').write_text(
    json.dumps({
        'family':GLOBAL_FAMILY,
        'model':GLOBAL_MODEL,
        'dev_cv_auprc':GLOBAL_SCORE,
        'complexity_margin':GLOBAL_COMPLEXITY_MARGIN,
    },indent=2)
)


In [ ]:
# ============================================================
# 19. METRIC HELPERS + DEV OOF THRESHOLDS
# ============================================================
def binary_metrics(y,p,threshold):
    y=np.asarray(y,dtype=int)
    p=np.asarray(p,dtype=float)
    pred=(p>=threshold).astype(int)

    return {
        'AUPRC':float(average_precision_score(y,p)),
        'AUROC':float(roc_auc_score(y,p)),
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall':float(recall_score(y,pred,zero_division=0)),
        'F1':float(f1_score(y,pred,zero_division=0)),
        'MCC':float(matthews_corrcoef(y,pred)),
    }

def best_mcc_threshold(y,p):
    best=(-999,None)

    for t in np.linspace(0.01,0.99,199):
        m=matthews_corrcoef(
            y,
            (p>=t).astype(int)
        )

        if m>best[0]:
            best=(m,float(t))

    return best[1],best[0]

dev_y=PACKS['DevSet1010']['y'].astype(int)

LOCAL_OOF=ARCH_OOF[LOCKED_ARCH]
LOCAL_THR,LOCAL_OOF_MCC=best_mcc_threshold(dev_y,LOCAL_OOF)

PATCH_OOF_LOCK=PATCH_OOF[BEST_PATCH]
PATCH_THR,PATCH_OOF_MCC=best_mcc_threshold(dev_y,PATCH_OOF_LOCK)

print('Local threshold:',LOCAL_THR,'OOF MCC:',LOCAL_OOF_MCC)
print('Patch threshold:',PATCH_THR,'OOF MCC:',PATCH_OOF_MCC)


In [ ]:
# ============================================================
# 20. FINAL EXTERNAL EVALUATION — LOCKED FAMILY WINNERS
# ============================================================
def predict_local_external(split):
    pack=PACKS[split]
    ds=MultiArrayDataset(pack['mods'],pack['y'])
    preds=[]

    for fold,state in ARCH_STATES[LOCKED_ARCH].items():
        model=ARCH_FACTORY[LOCKED_ARCH](DIMS).to(DEVICE)
        model.load_state_dict(state)
        preds.append(predict_flat(model,ds))
        del model
        torch.cuda.empty_cache()

    return np.mean(np.stack(preds),axis=0)

def predict_patch_external(split):
    ds=PatchProteinDataset(split,SPLIT_DF[split])
    preds=[]

    for fold,state in PATCH_STATES[BEST_PATCH].items():
        model=PATCH_FACTORY[BEST_PATCH](DIMS).to(DEVICE)
        model.load_state_dict(state)
        p,_=predict_patch(model,ds)
        preds.append(p)
        del model
        torch.cuda.empty_cache()

    return np.mean(np.stack(preds),axis=0)

rows=[]

if RUN_FINAL_EXTERNAL:
    for split in ['TestSet300','TestSetNew46']:
        y=PACKS[split]['y'].astype(int)

        lp=predict_local_external(split)
        pp=predict_patch_external(split)

        for family,model,p,thr,devscore in [
            (
                'LOCAL_MULTIMODAL',
                LOCKED_ARCH,
                lp,
                LOCAL_THR,
                LOCKED_ARCH_SCORE
            ),
            (
                'PATCH_HYPERGRAPH',
                BEST_PATCH,
                pp,
                PATCH_THR,
                BEST_PATCH_SCORE
            ),
        ]:
            m05=binary_metrics(y,p,0.5)
            moo=binary_metrics(y,p,thr)

            row={
                'family':family,
                'model':model,
                'dataset':split,
                'Dev_CV_AUPRC':devscore,
                'Dev_OOF_MCC_threshold':thr,
            }

            for k,v in m05.items():
                row[f'{k}_fixed0.5']=v

            for k,v in moo.items():
                row[f'{k}_OOFthr']=v

            rows.append(row)

        pd.DataFrame({
            'prot_id':PACKS[split]['pids'],
            'residue_index':PACKS[split]['ridx'],
            'y_true':y,
            'local_probability':lp,
            'patch_probability':pp,
        }).to_csv(
            RUN_ROOT/f'predictions_{split}.csv.gz',
            index=False
        )

external_df=pd.DataFrame(rows)
external_df.to_csv(RUN_ROOT/'external_family_winners.csv',index=False)

display(external_df)

headline=external_df[
    external_df.family.eq(GLOBAL_FAMILY)
    & external_df.model.eq(GLOBAL_MODEL)
].copy()

headline.to_csv(RUN_ROOT/'FINAL_HEADLINE_EXTERNAL.csv',index=False)

print('HEADLINE:')
display(headline)


In [ ]:
# ============================================================
# NUCLEAR V2-A. Surface-topology feature block (Dev-selected)
# Append AFTER V1 has finished. Does not overwrite V1.
# ============================================================
from pathlib import Path
from collections import defaultdict, Counter
import json, math, numpy as np, pandas as pd
from scipy.spatial import cKDTree

V2_RUN_ROOT = BASE/'results'/'NUCLEAR_V2_SELECTIVE_MULTISCALE'
V2_RUN_ROOT.mkdir(parents=True, exist_ok=True)

V2_CACHE_ROOT = BASE/'cache'/'nuclear_v2_selective_multiscale'
V2_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

V2_FORCE_FEATURES = False

V2_CHARGE = {'K':1.0,'R':1.0,'H':0.10,'D':-1.0,'E':-1.0}
V2_KR  = set('KR')
V2_KRH = set('KRH')
V2_DE  = set('DE')
V2_POL = set('NQSTY')
V2_HBD = set('KRHNQSTYW')

V2_BACKBONE = {'N','CA','C','O','OXT'}

def v2_rep_coord(res):
    side = [
        np.asarray(a.coord, dtype=np.float32)
        for a in res
        if a.element != 'H' and a.name not in V2_BACKBONE
    ]

    if side:
        return np.mean(np.stack(side), axis=0)

    if 'CA' in res:
        return np.asarray(res['CA'].coord, dtype=np.float32)

    atoms = [
        np.asarray(a.coord, dtype=np.float32)
        for a in res
    ]

    return np.mean(np.stack(atoms), axis=0)


def v2_coords(split, pid, seq):
    p = PDB_NORM_ROOT / split / f'{pid}.pdb'

    st = PDBParser(QUIET=True).get_structure(
        'v2',
        str(p)
    )

    m = next(st.get_models())
    rr = standard_residues(m['A'])

    if residue_sequence(rr) != seq:
        raise RuntimeError(
            f'{split}/{pid}: PDB/sequence mismatch'
        )

    return np.stack(
        [v2_rep_coord(r) for r in rr],
        axis=0
    ).astype(np.float32)


def v2_exposure_proxy(coords):
    """
    Label-free geometric exposure proxy.

    Fewer residues within 10 Å = more surface exposed.
    Returns [0,1], ranked within each protein.
    """

    tree = cKDTree(coords)

    n10 = np.array(
        [
            max(
                len(
                    tree.query_ball_point(
                        coords[i],
                        10.0
                    )
                ) - 1,
                0
            )
            for i in range(len(coords))
        ],
        dtype=np.float32
    )

    if len(coords) <= 1:
        return (
            np.ones(
                len(coords),
                dtype=np.float32
            ),
            n10
        )

    order = np.argsort(
        np.argsort(n10)
    )

    dense_rank = (
        order.astype(np.float32)
        / (len(coords) - 1)
    )

    exposure = 1.0 - dense_rank

    return (
        exposure.astype(np.float32),
        n10
    )


def v2_component_sizes(
    indices,
    coords,
    threshold=8.0
):
    if len(indices) == 0:
        return {}

    idx = np.asarray(
        indices,
        dtype=int
    )

    cc = np.asarray(
        coords,
        dtype=np.float32
    )

    parent = list(
        range(len(idx))
    )

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)

        if ra != rb:
            parent[rb] = ra

    if len(idx) > 1:
        for a, b in cKDTree(
            cc
        ).query_pairs(
            threshold
        ):
            union(a, b)

    groups = defaultdict(list)

    for j, rid in enumerate(idx):
        groups[
            find(j)
        ].append(
            int(rid)
        )

    out = {}

    for g in groups.values():
        for rid in g:
            out[rid] = len(g)

    return out


def build_na_surftopo16(
    split,
    pid,
    seq
):
    coords = v2_coords(
        split,
        pid,
        seq
    )

    tree = cKDTree(
        coords
    )

    exposure, n10 = v2_exposure_proxy(
        coords
    )

    L = len(seq)

    q = np.array(
        [
            V2_CHARGE.get(
                a,
                0.0
            )
            for a in seq
        ],
        dtype=np.float32
    )

    # top ~50% geometrically exposed residues
    surface_mask = (
        exposure >= 0.50
    )

    surf_basic_idx = [
        i
        for i, a in enumerate(seq)
        if (
            surface_mask[i]
            and a in V2_KRH
        )
    ]

    comp = v2_component_sizes(
        surf_basic_idx,
        (
            coords[surf_basic_idx]
            if surf_basic_idx
            else np.zeros(
                (0, 3),
                np.float32
            )
        ),
        threshold=8.0
    )

    out = np.zeros(
        (L, 16),
        dtype=np.float32
    )

    for i in range(L):

        ids8 = np.asarray(
            tree.query_ball_point(
                coords[i],
                8.0
            ),
            dtype=int
        )

        ids8 = ids8[
            ids8 != i
        ]

        surf8 = (
            ids8[
                surface_mask[ids8]
            ]
            if len(ids8)
            else np.array(
                [],
                dtype=int
            )
        )

        # 0: geometric exposure
        out[i, 0] = exposure[i]

        # 1: packing density proxy
        out[i, 1] = min(
            float(n10[i]) / 40.0,
            1.0
        )

        # 2: fraction of local neighbors that are surface-like
        out[i, 2] = (
            len(surf8)
            / max(
                len(ids8),
                1
            )
        )

        if len(surf8):

            aa = [
                seq[j]
                for j in surf8
            ]

            qq = q[
                surf8
            ]

            dd = np.linalg.norm(
                coords[surf8]
                - coords[i][None, :],
                axis=1
            )

            # 3-6: surface chemistry
            out[i, 3] = (
                sum(
                    a in V2_KRH
                    for a in aa
                )
                / len(aa)
            )

            out[i, 4] = (
                sum(
                    a in V2_KR
                    for a in aa
                )
                / len(aa)
            )

            out[i, 5] = (
                sum(
                    a in V2_POL
                    for a in aa
                )
                / len(aa)
            )

            out[i, 6] = (
                sum(
                    a in V2_DE
                    for a in aa
                )
                / len(aa)
            )

            # 7: mean surface charge
            out[i, 7] = float(
                np.mean(qq)
            )

            # 8: distance-weighted electrostatic proxy
            out[i, 8] = float(
                np.tanh(
                    np.sum(
                        qq / (dd + 1.0)
                    )
                )
            )

            # 12: charge heterogeneity
            out[i, 12] = float(
                np.std(qq)
            )

            # 13: basic-acidic surface imbalance
            out[i, 13] = float(
                np.tanh(
                    (
                        sum(
                            a in V2_KRH
                            for a in aa
                        )
                        -
                        sum(
                            a in V2_DE
                            for a in aa
                        )
                    )
                    / 5.0
                )
            )

            # 14: H-bond donor-rich surface
            out[i, 14] = (
                sum(
                    a in V2_HBD
                    for a in aa
                )
                / len(aa)
            )

            # 15: tertiary / nonlocal surface contacts
            out[i, 15] = float(
                np.mean(
                    np.abs(
                        surf8 - i
                    ) > 10
                )
            )

        # 9: nearest exposed basic residue distance
        basic = [
            j
            for j in surf_basic_idx
            if j != i
        ]

        if basic:
            d = np.linalg.norm(
                coords[basic]
                - coords[i][None, :],
                axis=1
            )

            out[i, 9] = min(
                float(d.min()) / 20.0,
                1.0
            )

        else:
            out[i, 9] = 1.0

        # 10: exposed basic connected-component size
        out[i, 10] = min(
            comp.get(
                i,
                0
            ),
            20
        ) / 20.0

        # 11: distance to exposed positive centroid
        ids12 = np.asarray(
            tree.query_ball_point(
                coords[i],
                12.0
            ),
            dtype=int
        )

        pos = [
            j
            for j in ids12
            if (
                surface_mask[j]
                and seq[j] in V2_KRH
            )
        ]

        if pos:

            cent = coords[
                pos
            ].mean(
                axis=0
            )

            out[i, 11] = min(
                float(
                    np.linalg.norm(
                        coords[i] - cent
                    )
                )
                / 20.0,
                1.0
            )

        else:
            out[i, 11] = 1.0

    return out


V2_SURF_CACHE = (
    V2_CACHE_ROOT
    / 'surftopo16'
)

for split, df in SPLIT_DF.items():

    od = (
        V2_SURF_CACHE
        / split
    )

    od.mkdir(
        parents=True,
        exist_ok=True
    )

    store = ctx.stores[
        STORE_KEY[split]
    ]

    for n, r in enumerate(
        df.itertuples(),
        start=1
    ):

        fp = (
            od
            / f'{r.prot_id}.npy'
        )

        good = False

        if (
            fp.exists()
            and not V2_FORCE_FEATURES
        ):

            try:

                a = np.load(
                    fp,
                    mmap_mode='r'
                )

                good = (
                    a.shape
                    ==
                    (
                        len(r.sequence),
                        16
                    )
                )

            except Exception:
                good = False

        if not good:

            a = build_na_surftopo16(
                split,
                str(r.prot_id),
                r.sequence
            )

            np.save(
                fp,
                a.astype(
                    np.float32
                )
            )

        else:

            a = np.load(
                fp
            ).astype(
                np.float32
            )

        set_block(
            store,
            r.prot_id,
            'NA_SURFTOPO16',
            a
        )

        if n % 250 == 0:
            print(
                split,
                n,
                '/',
                len(df)
            )

print(
    'NA_SURFTOPO16 attached: PASS'
)

In [ ]:
# ============================================================
# NUCLEAR V2-B. Dev-only representation re-lock
# ============================================================

V2_REP_GRID = {

    'V2_ESM': [
        'ESM'
    ],

    # exact V1 winner/control
    'V2_V1_PATCHORG': [
        'ESM',
        'ACCESS',
        'NA_PATCH3D40',
        'NA_TOPO12'
    ],

    # new feature alone
    'V2_ESM_SURFTOPO': [
        'ESM',
        'NA_SURFTOPO16'
    ],

    # V1 winner + new topology
    'V2_PATCHORG_SURFTOPO': [
        'ESM',
        'ACCESS',
        'NA_PATCH3D40',
        'NA_TOPO12',
        'NA_SURFTOPO16'
    ],

    # see whether explicit electrostatics comes back
    # once surface topology is available
    'V2_PATCHORG_SURFTOPO_ELEC': [
        'ESM',
        'ACCESS',
        'ELEC3D',
        'NA_PATCH3D40',
        'NA_TOPO12',
        'NA_SURFTOPO16'
    ],
}


v2_rep_cv, v2_rep_folds = (
    suite.run_dev_cv_only(
        ctx,
        V2_REP_GRID,
        models=('MLP',),
        run_version='nuclear_v2_01_surftopo_rep',
    )
)


v2_rep_cv.to_csv(
    V2_RUN_ROOT
    / 'representation_cv.csv',
    index=False
)

v2_rep_folds.to_csv(
    V2_RUN_ROOT
    / 'representation_folds.csv',
    index=False
)


display(
    v2_rep_cv.sort_values(
        'CV_AUPRC',
        ascending=False
    )
)


V2_REP_NAME = str(
    v2_rep_cv
    .sort_values(
        'CV_AUPRC',
        ascending=False
    )
    .iloc[0]
    .representation
)


V2_BLOCKS = list(
    V2_REP_GRID[
        V2_REP_NAME
    ]
)


V2_REP_SCORE = float(
    v2_rep_cv.loc[
        v2_rep_cv
        .representation
        .eq(
            V2_REP_NAME
        ),
        'CV_AUPRC'
    ].iloc[0]
)


print(
    'V2 LOCKED REPRESENTATION:',
    V2_REP_NAME
)

print(
    'V2 BLOCKS:',
    V2_BLOCKS
)

print(
    'V2 Dev CV AUPRC:',
    V2_REP_SCORE
)


def v2_modality_spec(
    blocks
):

    spec = {}

    if 'ESM' in blocks:
        spec['ESM'] = [
            'ESM'
        ]

    surface = [
        b
        for b in [
            'ACCESS',
            'NA_SURFTOPO16'
        ]
        if b in blocks
    ]

    if surface:
        spec[
            'SURFACE'
        ] = surface

    patch = [
        b
        for b in [
            'NA_PATCH3D40',
            'NA_TOPO12'
        ]
        if b in blocks
    ]

    if patch:
        spec[
            'PATCH_LOCAL'
        ] = patch

    electro = [
        b
        for b in [
            'ELEC3D',
            'NA_SEQELECTRO32'
        ]
        if b in blocks
    ]

    if electro:
        spec[
            'ELECTRO'
        ] = electro

    chem = [
        b
        for b in [
            'NA_HBOND16'
        ]
        if b in blocks
    ]

    if chem:
        spec[
            'CHEM'
        ] = chem

    assigned = {
        x
        for v in spec.values()
        for x in v
    }

    other = [
        b
        for b in blocks
        if b not in assigned
    ]

    if other:
        spec[
            'OTHER'
        ] = other

    return spec


V2_MODALITY_SPEC = (
    v2_modality_spec(
        V2_BLOCKS
    )
)


def v2_concat_blocks(
    store,
    pid,
    blocks
):

    xs = [
        get_block(
            store,
            pid,
            b
        ).astype(
            np.float32,
            copy=False
        )

        for b in blocks
    ]

    return np.concatenate(
        xs,
        axis=1
    )


V2_DIMS = {}


for m, blocks in (
    V2_MODALITY_SPEC.items()
):

    sample_pid = str(
        train_df
        .iloc[0]
        .prot_id
    )

    V2_DIMS[m] = (
        v2_concat_blocks(
            ctx.stores[
                'train'
            ],
            sample_pid,
            blocks
        )
        .shape[1]
    )


print(
    'V2 modalities:',
    V2_MODALITY_SPEC
)

print(
    'V2 dims:',
    V2_DIMS
)


(
    V2_RUN_ROOT
    / 'locked_representation.json'
).write_text(
    json.dumps(
        {
            'representation':
                V2_REP_NAME,

            'blocks':
                V2_BLOCKS,

            'modalities':
                V2_MODALITY_SPEC,

            'dev_cv_auprc':
                V2_REP_SCORE,
        },
        indent=2
    )
)

In [ ]:
# ============================================================
# NUCLEAR V2-C. Selective multi-scale surface patches
# ============================================================

V2_RADII = (
    6.0,
    8.0,
    10.0,
    12.0
)

V2_PATCH_MAX = 48

V2_JACCARD_NMS = 0.65

V2_FORCE_PATCH = False


V2_PATCH_CACHE = (
    V2_CACHE_ROOT
    / 'selective_multiscale_patches'
)

V2_PATCH_CACHE.mkdir(
    parents=True,
    exist_ok=True
)


def v2_patch_feature(
    seq,
    coords,
    exposure,
    members,
    radius
):

    members = np.asarray(
        members,
        dtype=int
    )

    aa = [
        seq[j]
        for j in members
    ]

    cc = coords[
        members
    ]

    qq = np.asarray(
        [
            V2_CHARGE.get(
                a,
                0.0
            )
            for a in aa
        ],
        dtype=np.float32
    )

    n = len(
        members
    )


    cent = cc.mean(
        axis=0
    )

    radial = np.linalg.norm(
        cc
        - cent[None, :],
        axis=1
    )


    pair = []
    seqsep = []

    for x in range(n):

        for y in range(
            x + 1,
            n
        ):

            pair.append(
                float(
                    np.linalg.norm(
                        cc[x]
                        -
                        cc[y]
                    )
                )
            )

            seqsep.append(
                abs(
                    int(
                        members[x]
                    )
                    -
                    int(
                        members[y]
                    )
                )
            )


    pair = np.asarray(
        (
            pair
            if pair
            else [0.0]
        ),
        dtype=np.float32
    )

    seqsep = np.asarray(
        (
            seqsep
            if seqsep
            else [0.0]
        ),
        dtype=np.float32
    )


    comp = Counter(
        aa
    )

    probs = np.asarray(
        [
            c / n
            for c in comp.values()
        ],
        dtype=np.float32
    )


    entropy = float(
        -np.sum(
            probs
            *
            np.log(
                probs
                + 1e-12
            )
        )
        /
        math.log(20)
    )


    f = np.zeros(
        24,
        dtype=np.float32
    )


    # scale
    f[0] = (
        radius
        / 12.0
    )

    # size
    f[1] = (
        min(
            n,
            40
        )
        / 40.0
    )

    # surface exposure
    f[2] = float(
        np.mean(
            exposure[
                members
            ]
        )
    )

    f[3] = float(
        np.min(
            exposure[
                members
            ]
        )
    )

    # chemistry
    f[4] = (
        sum(
            a in V2_KR
            for a in aa
        )
        / n
    )

    f[5] = (
        sum(
            a in V2_KRH
            for a in aa
        )
        / n
    )

    f[6] = (
        sum(
            a in V2_POL
            for a in aa
        )
        / n
    )

    f[7] = (
        sum(
            a in V2_DE
            for a in aa
        )
        / n
    )

    f[8] = (
        sum(
            a in V2_HBD
            for a in aa
        )
        / n
    )

    f[9] = float(
        np.mean(qq)
    )

    f[10] = float(
        np.std(qq)
    )

    f[11] = float(
        np.tanh(
            (
                sum(
                    a in V2_KRH
                    for a in aa
                )
                -
                sum(
                    a in V2_DE
                    for a in aa
                )
            )
            / 5.0
        )
    )

    # geometry
    f[12] = min(
        float(
            radial.mean()
        )
        / 12.0,
        1.0
    )

    f[13] = min(
        float(
            radial.std()
        )
        / 8.0,
        1.0
    )

    f[14] = min(
        float(
            radial.max()
        )
        / 15.0,
        1.0
    )

    f[15] = min(
        float(
            pair.mean()
        )
        / 15.0,
        1.0
    )

    f[16] = min(
        float(
            pair.std()
        )
        / 10.0,
        1.0
    )

    # tertiary contacts
    f[17] = float(
        np.mean(
            seqsep > 10
        )
    )

    f[18] = min(
        float(
            seqsep.mean()
        )
        / 50.0,
        1.0
    )

    # chemistry diversity
    f[19] = entropy


    if n >= 3:

        z = (
            cc
            -
            cc.mean(
                axis=0,
                keepdims=True
            )
        )

        cov = (
            z.T
            @ z
            / max(
                n - 1,
                1
            )
        )

        eig = np.sort(
            np.clip(
                np.linalg.eigvalsh(
                    cov
                ),
                0,
                None
            )
        )[::-1]

        total = float(
            eig.sum()
            + 1e-8
        )

        # linearity
        f[20] = np.clip(
            (
                eig[0]
                -
                eig[1]
            )
            /
            (
                eig[0]
                + 1e-8
            ),
            0,
            1
        )

        # planarity
        f[21] = np.clip(
            (
                eig[1]
                -
                eig[2]
            )
            /
            (
                eig[0]
                + 1e-8
            ),
            0,
            1
        )

        # isotropy
        f[22] = np.clip(
            eig[2]
            / total,
            0,
            1
        )


    # broad sequence dispersion
    f[23] = float(
        np.mean(
            np.abs(
                members
                -
                int(
                    np.median(
                        members
                    )
                )
            )
            > 10
        )
    )


    return f


def v2_jaccard(
    a,
    b
):

    a = set(a)
    b = set(b)

    return (
        len(
            a & b
        )
        /
        max(
            len(
                a | b
            ),
            1
        )
    )


def build_v2_patch_graph(
    split,
    pid,
    seq
):

    coords = v2_coords(
        split,
        pid,
        seq
    )

    tree = cKDTree(
        coords
    )

    exposure, _ = (
        v2_exposure_proxy(
            coords
        )
    )


    candidates = []


    for seed, a in enumerate(
        seq
    ):

        # Basic seeds may be only moderately exposed.
        if a in V2_KRH:

            if exposure[
                seed
            ] < 0.35:

                continue

        # Polar-only seeds must be strongly exposed.
        elif a in V2_POL:

            if exposure[
                seed
            ] < 0.65:

                continue

        else:
            continue


        for radius in V2_RADII:

            raw = (
                tree.query_ball_point(
                    coords[seed],
                    radius
                )
            )

            # mostly surface / near-surface residues
            members = tuple(
                sorted(
                    j
                    for j in raw
                    if exposure[j] >= 0.25
                )
            )


            if len(
                members
            ) < 4:
                continue


            if not any(
                seq[j] in V2_KRH
                for j in members
            ):
                continue


            feat = v2_patch_feature(
                seq,
                coords,
                exposure,
                members,
                radius
            )


            score = (
                2.0
                * feat[5]

                + 0.70
                * feat[6]

                - 0.70
                * feat[7]

                + 1.20
                * feat[2]

                + 0.25
                * feat[17]

                - 0.10
                * feat[0]
            )


            candidates.append(
                (
                    float(
                        score
                    ),
                    members,
                    feat
                )
            )


    # exact duplicate removal
    unique = {}


    for (
        score,
        members,
        feat
    ) in candidates:

        if (
            members not in unique
            or score
            >
            unique[
                members
            ][0]
        ):

            unique[
                members
            ] = (
                score,
                feat
            )


    ranked = sorted(
        [
            (
                s,
                m,
                f
            )
            for m, (
                s,
                f
            )
            in unique.items()
        ],
        key=lambda x:
            x[0],
        reverse=True
    )


    # Jaccard non-maximum suppression
    selected = []


    for (
        score,
        members,
        feat
    ) in ranked:

        if any(
            v2_jaccard(
                members,
                m2
            )
            >=
            V2_JACCARD_NMS

            for _, m2, _
            in selected
        ):
            continue


        selected.append(
            (
                score,
                members,
                feat
            )
        )


        if (
            len(
                selected
            )
            >=
            V2_PATCH_MAX
        ):
            break


    edge_res = []
    edge_patch = []
    patch_feat = []


    for pidx, (
        _,
        members,
        feat
    ) in enumerate(
        selected
    ):

        patch_feat.append(
            feat
        )


        for rid in members:

            edge_res.append(
                int(rid)
            )

            edge_patch.append(
                int(pidx)
            )


    return (
        np.asarray(
            edge_res,
            dtype=np.int64
        ),

        np.asarray(
            edge_patch,
            dtype=np.int64
        ),

        np.asarray(
            patch_feat,
            dtype=np.float32
        ).reshape(
            -1,
            24
        )
    )


V2_PATCH_RAM = {}

v2_audit = []


for split, df in SPLIT_DF.items():

    od = (
        V2_PATCH_CACHE
        / split
    )

    od.mkdir(
        parents=True,
        exist_ok=True
    )


    for n, r in enumerate(
        df.itertuples(),
        start=1
    ):

        fp = (
            od
            / f'{r.prot_id}.npz'
        )


        good = False


        if (
            fp.exists()
            and not V2_FORCE_PATCH
        ):

            try:

                z = np.load(
                    fp
                )

                good = (
                    z[
                        'patch_feat'
                    ].ndim
                    == 2

                    and
                    z[
                        'patch_feat'
                    ].shape[1]
                    == 24

                    and
                    len(
                        z[
                            'edge_res'
                        ]
                    )
                    ==
                    len(
                        z[
                            'edge_patch'
                        ]
                    )
                )

            except Exception:
                good = False


        if not good:

            er, ep, pf = (
                build_v2_patch_graph(
                    split,
                    str(
                        r.prot_id
                    ),
                    r.sequence
                )
            )


            np.savez_compressed(
                fp,
                edge_res=er,
                edge_patch=ep,
                patch_feat=pf
            )


        z = np.load(
            fp
        )


        er = z[
            'edge_res'
        ].astype(
            np.int64
        )


        ep = z[
            'edge_patch'
        ].astype(
            np.int64
        )


        pf = z[
            'patch_feat'
        ].astype(
            np.float32
        )


        V2_PATCH_RAM[
            (
                split,
                str(
                    r.prot_id
                )
            )
        ] = (
            er,
            ep,
            pf
        )


        covered = (
            len(
                np.unique(
                    er
                )
            )
            if len(er)
            else 0
        )


        v2_audit.append(
            {
                'dataset':
                    split,

                'prot_id':
                    str(
                        r.prot_id
                    ),

                'n_residues':
                    len(
                        r.sequence
                    ),

                'n_patches':
                    len(pf),

                'n_memberships':
                    len(er),

                'coverage':
                    covered
                    /
                    max(
                        len(
                            r.sequence
                        ),
                        1
                    ),
            }
        )


        if n % 250 == 0:

            print(
                split,
                n,
                '/',
                len(df)
            )


v2_patch_audit = pd.DataFrame(
    v2_audit
)


v2_patch_audit.to_csv(
    V2_RUN_ROOT
    / 'patch_audit.csv',
    index=False
)


display(
    v2_patch_audit
    .groupby(
        'dataset'
    )
    .agg(
        mean_patches=(
            'n_patches',
            'mean'
        ),

        median_patches=(
            'n_patches',
            'median'
        ),

        mean_coverage=(
            'coverage',
            'mean'
        ),

        median_coverage=(
            'coverage',
            'median'
        ),

        mean_memberships=(
            'n_memberships',
            'mean'
        ),
    )
    .reset_index()
)

In [ ]:
# ============================================================
# NUCLEAR FINAL EXPERIMENT
# V1 Attention architecture + V2 selective multi-scale patches
# One-cell, Dev-lock first, then external evaluation
# ============================================================
from pathlib import Path
import json, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    matthews_corrcoef, precision_score, recall_score
)

assert 'V2_PATCH_RAM' in globals(), \
    "Run Nuclear V2-C first to create V2_PATCH_RAM."

assert 'NuclearPatchHypergraph' in globals()
assert 'PatchProteinDataset' in globals()
assert 'run_patch_cv' in globals()

NUC_MS_ROOT = (
    BASE
    / 'results'
    / 'NUCLEAR_V1_ATTENTION_V2_MULTISCALE_PATCH'
)

NUC_MS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Same V1 hypergraph architecture.
# ONLY patch static dimension changes 18 -> 24
# because V2 multiscale patches have 24 features.
# ============================================================

class NuclearPatchHypergraphMS(
    NuclearPatchHypergraph
):
    def __init__(
        self,
        dims,
        mode='attention',
        latent=192,
        drop=0.2
    ):
        super().__init__(
            dims,
            mode=mode,
            latent=latent,
            moe=False,
            drop=drop
        )

        self.patch_static = nn.Sequential(
            nn.LayerNorm(24),
            nn.Linear(24, 64),
            nn.GELU()
        )


def collate_patch_ms(batch):

    keys = sorted(
        batch[0]['mods']
    )

    mods = {
        k: torch.from_numpy(
            np.concatenate(
                [
                    b['mods'][k]
                    for b in batch
                ],
                axis=0
            )
        ).float()

        for k in keys
    }

    y = torch.from_numpy(
        np.concatenate(
            [
                b['y']
                for b in batch
            ]
        )
    ).float()

    patch_feat = []
    edge_res = []
    edge_patch = []

    roff = 0
    poff = 0

    for b in batch:

        L = len(
            b['y']
        )

        P = len(
            b['patch_feat']
        )

        if P:
            patch_feat.append(
                torch.from_numpy(
                    b['patch_feat']
                ).float()
            )

        if len(
            b['edge_res']
        ):

            edge_res.append(
                torch.from_numpy(
                    b['edge_res']
                ).long()
                + roff
            )

            edge_patch.append(
                torch.from_numpy(
                    b['edge_patch']
                ).long()
                + poff
            )

        roff += L
        poff += P

    patch_feat = (
        torch.cat(
            patch_feat,
            dim=0
        )
        if patch_feat
        else torch.zeros(
            (0, 24),
            dtype=torch.float32
        )
    )

    edge_res = (
        torch.cat(
            edge_res,
            dim=0
        )
        if edge_res
        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )

    edge_patch = (
        torch.cat(
            edge_patch,
            dim=0
        )
        if edge_patch
        else torch.zeros(
            (0,),
            dtype=torch.long
        )
    )

    return {
        'mods': mods,
        'y': y,
        'edge_res': edge_res,
        'edge_patch': edge_patch,
        'patch_feat': patch_feat,
    }


# ============================================================
# Save current V1 globals.
# Temporarily substitute ONLY:
#   patch bank
#   patch-static dimensionality
# ============================================================

_OLD_PATCH_RAM = PATCH_RAM
_OLD_PATCH_FACTORY = PATCH_FACTORY
_OLD_PATCH_STATE_ROOT = PATCH_STATE_ROOT
_OLD_COLLATE_PATCH = collate_patch


PATCH_RAM = V2_PATCH_RAM

PATCH_FACTORY = {
    'NA_PatchHG_Attention_MS':
        lambda dims:
            NuclearPatchHypergraphMS(
                dims,
                mode='attention',
                latent=PATCH_LATENT,
                drop=0.2
            )
}

PATCH_STATE_ROOT = (
    NUC_MS_ROOT
    / 'states'
)

PATCH_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

collate_patch = collate_patch_ms


try:

    print("=" * 100)
    print(
        "NUCLEAR: V1 ATTENTION "
        "+ V2 SELECTIVE MULTI-SCALE PATCHES"
    )
    print("=" * 100)

    # ========================================================
    # DEV CV
    # ========================================================

    (
        ms_summary,
        ms_folds,
        ms_oof,
        ms_states
    ) = run_patch_cv(
        'NA_PatchHG_Attention_MS'
    )

    ms_summary_df = pd.DataFrame(
        [ms_summary]
    )

    display(
        ms_summary_df
    )

    display(
        ms_folds
    )

    ms_summary_df.to_csv(
        NUC_MS_ROOT
        / 'dev_cv_summary.csv',
        index=False
    )

    ms_folds.to_csv(
        NUC_MS_ROOT
        / 'dev_cv_folds.csv',
        index=False
    )


    MS_DEV = float(
        ms_summary[
            'CV_AUPRC_mean'
        ]
    )


    # V1 original Attention performance
    V1_DEV = float(
        patch_summary.loc[
            patch_summary
            .architecture
            .eq(
                'NA_PatchHG_Attention'
            ),
            'CV_AUPRC_mean'
        ].iloc[0]
    )


    print(
        f"\nV1 Attention Dev mean AUPRC : "
        f"{V1_DEV:.6f}"
    )

    print(
        f"MS Attention Dev mean AUPRC : "
        f"{MS_DEV:.6f}"
    )

    print(
        f"Delta MS - V1               : "
        f"{MS_DEV - V1_DEV:+.6f}"
    )


    # Complexity-aware stop rule
    MS_WINS = (
        MS_DEV
        - V1_DEV
        >= 0.002
    )

    print(
        "MS becomes new headline?     :",
        MS_WINS
    )


    # ========================================================
    # Frozen DEV OOF threshold
    # ========================================================

    dev_y = np.concatenate(
        train_df
        .nuclear_binding
        .apply(
            lambda x:
                np.asarray(
                    x,
                    dtype=np.int8
                )
        )
        .tolist()
    )


    (
        ms_thr,
        ms_oof_mcc
    ) = best_mcc_threshold(
        dev_y,
        ms_oof
    )


    print(
        "Frozen Dev threshold:",
        ms_thr
    )

    print(
        "Dev OOF MCC:",
        ms_oof_mcc
    )


    # ========================================================
    # External prediction
    # ========================================================

    def _predict_ms_external(
        split
    ):

        ds = PatchProteinDataset(
            split,
            SPLIT_DF[
                split
            ]
        )

        preds = []

        for fold, state in (
            ms_states.items()
        ):

            model = PATCH_FACTORY[
                'NA_PatchHG_Attention_MS'
            ](
                DIMS
            ).to(
                DEVICE
            )

            model.load_state_dict(
                state
            )

            p, _ = predict_patch(
                model,
                ds
            )

            preds.append(
                p
            )

            del model

            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()

        return np.mean(
            np.stack(
                preds
            ),
            axis=0
        )


    ext_rows = []


    for split in [
        'TestSet300',
        'TestSetNew46'
    ]:

        y = np.concatenate(
            SPLIT_DF[
                split
            ]
            .nuclear_binding
            .apply(
                lambda x:
                    np.asarray(
                        x,
                        dtype=np.int8
                    )
            )
            .tolist()
        )

        p = _predict_ms_external(
            split
        )


        m05 = binary_metrics(
            y,
            p,
            0.5
        )

        mdev = binary_metrics(
            y,
            p,
            ms_thr
        )


        row = {
            'dataset':
                split,

            'method':
                'NA_PatchHG_Attention_MS',

            'Dev_CV_AUPRC':
                MS_DEV,

            'Dev_OOF_MCC_threshold':
                ms_thr,
        }


        row.update(
            {
                f'{k}_fixed0.5':
                    v

                for k, v
                in m05.items()
            }
        )


        row.update(
            {
                f'{k}_OOFthr':
                    v

                for k, v
                in mdev.items()
            }
        )


        ext_rows.append(
            row
        )


    ms_external = pd.DataFrame(
        ext_rows
    )


    display(
        ms_external
    )


    ms_external.to_csv(
        NUC_MS_ROOT
        / 'external_metrics.csv',
        index=False
    )


    # ========================================================
    # FINAL comparison:
    # new MS vs V1 vs bindNode24 vs bindEmbed21DL
    # ========================================================

    cmp_rows = []


    for _, r in (
        ms_external.iterrows()
    ):

        cmp_rows.append(
            {
                'dataset':
                    r['dataset'],

                'method':
                    'MS selective PatchHG Attention',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


    # Original V1
    for _, r in (
        headline.iterrows()
    ):

        cmp_rows.append(
            {
                'dataset':
                    r['dataset'],

                'method':
                    f'V1 {GLOBAL_MODEL}',

                'AUPRC':
                    r[
                        'AUPRC_fixed0.5'
                    ],

                'AUROC':
                    r[
                        'AUROC_fixed0.5'
                    ],

                'F1':
                    r[
                        'F1_fixed0.5'
                    ],

                'MCC':
                    r[
                        'MCC_fixed0.5'
                    ],
            }
        )


    baseline_fp = (
        BASELINE_WORK
        / 'results'
        / 'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
    )


    if baseline_fp.exists():

        bb = pd.read_csv(
            baseline_fp
        )


        if 'label' in bb.columns:

            bb = bb[
                bb.label.eq(
                    'nuclear_binding'
                )
            ].copy()


        if (
            'operating_point'
            in bb.columns
        ):

            bb = bb[
                bb.operating_point.eq(
                    'fixed_0.5'
                )
            ].copy()


        for _, r in (
            bb.iterrows()
        ):

            if str(
                r.method
            ) not in {
                'bindNode24',
                'bindEmbed21DL'
            }:
                continue


            apcol = (
                'AP_AUPRC'
                if 'AP_AUPRC'
                in bb.columns
                else 'AUPRC'
            )


            cmp_rows.append(
                {
                    'dataset':
                        str(
                            r.dataset
                        ),

                    'method':
                        str(
                            r.method
                        ),

                    'AUPRC':
                        float(
                            r[
                                apcol
                            ]
                        ),

                    'AUROC':
                        float(
                            r.AUROC
                        ),

                    'F1':
                        float(
                            r.global_positive_F1
                        ),

                    'MCC':
                        float(
                            r.global_MCC
                        ),
                }
            )


    ms_compare = (
        pd.DataFrame(
            cmp_rows
        )
        .sort_values(
            [
                'dataset',
                'AUPRC'
            ],
            ascending=[
                True,
                False
            ]
        )
    )


    print(
        "\nFINAL NUCLEAR COMPARISON"
    )

    display(
        ms_compare
    )


    ms_compare.to_csv(
        NUC_MS_ROOT
        / 'FINAL_COMPARISON.csv',
        index=False
    )


finally:

    # Restore V1 globals.
    PATCH_RAM = (
        _OLD_PATCH_RAM
    )

    PATCH_FACTORY = (
        _OLD_PATCH_FACTORY
    )

    PATCH_STATE_ROOT = (
        _OLD_PATCH_STATE_ROOT
    )

    collate_patch = (
        _OLD_COLLATE_PATCH
    )


print(
    "\nSaved:",
    NUC_MS_ROOT
)

# Part 2 — V3: re-evaluate the final model, improve it, bootstrap everything


In [ ]:
# ============================================================
# V3-0. Task configuration (names of the original notebook's objects)
# ============================================================
V3 = {
    "TASK": "NUCLEAR",
    "LABEL": "nuclear_binding",
    "V1": "NA_PatchHG_Attention",
    "MS": "NA_PatchHG_Attention_MS",
    "RAM": "PATCH_RAM",
    "MS_RAM": "V2_PATCH_RAM",
    "FACTORY": "PATCH_FACTORY",
    "STATE_ROOT": "PATCH_STATE_ROOT",
    "COLLATE": "collate_patch",
    "MS_COLLATE": "collate_patch_ms",
    "DATASET": "PatchProteinDataset",
    "PREDICT": "predict_patch",
    "RUN_CV": "run_patch_cv",
    "MS_CLASS": "NuclearPatchHypergraphMS",
    "LATENT": "PATCH_LATENT",
    "V1_OOF": "PATCH_OOF",
    "V1_STATES": "PATCH_STATES",
    "MS_OOF": "ms_oof",
    "MS_STATES": "ms_states",
    "ORIG_W_V1": 0.7,
    "ORIG_DEV": 0.5089611583669623,
    "ORIG_V1_DEV": 0.5019338304768628,
    "ORIG_EXT": {
        "TestSet300": 0.432839,
        "TestSetNew46": 0.124501
    }
}


In [ ]:
# ============================================================
# V3-1. Setup: helpers, bootstrap (B = 10000), member store
#       Everything below runs AFTER the original cells above (all of them REUSE their Drive caches/states).
# ============================================================
import re, time, hashlib
from contextlib import contextmanager
from sklearn.metrics import average_precision_score, roc_auc_score

G = globals()
LABEL = V3['LABEL']
SPLITS_EXT = ('TestSet300', 'TestSetNew46')
B_BOOT, BOOT_SEED = 10000, 2026
MARGIN = 0.002   # complexity margin of the original notebooks
V3_ROOT = BASE/'results'/f"{V3['TASK']}_V3_IMPROVE"
PRED_DIR = V3_ROOT/'member_preds'
PRED_DIR.mkdir(parents=True, exist_ok=True)

# originals captured once (the cells above restored them after the MS run)
ORIG_FACTORY = dict(G[V3['FACTORY']])
ORIG_SPEC = {k: list(v) for k, v in MODALITY_SPEC.items()}
ORIG_SEED = SEED
ORIG_REP_NAME = BEST_REP_NAME
PID0 = str(train_df.iloc[0].prot_id)

def _labels(split):
    return np.concatenate(SPLIT_DF[split][LABEL].apply(lambda x: np.asarray(x, np.int8)).tolist()).astype(np.int64)

Y = {s: _labels(s) for s in SPLIT_DF}                       # DevSet1010 in train_df (= OOF) order
PROT = {s: np.repeat(np.arange(len(df)), df.sequence.str.len().to_numpy()) for s, df in SPLIT_DF.items()}
_off, _cur = {}, 0
for r in train_df.itertuples():
    _off[str(r.prot_id)] = (_cur, _cur + len(r.sequence)); _cur += len(r.sequence)
FOLD_IDX = {f: np.concatenate([np.arange(*_off[str(p)]) for p in ids]) for f, ids in OFFICIAL_FOLDS.items()}

def fold_aps(p):
    return np.array([average_precision_score(Y['DevSet1010'][i], p[i]) for i in FOLD_IDX.values()])

def fold_mean(p):
    return float(fold_aps(p).mean())

@contextmanager
def swapped(**kv):
    old = {k: G[k] for k in kv}
    G.update(kv)
    try:
        yield
    finally:
        G.update(old)

def env_for(ms=False, extra=(), seed=None, name=None):
    """Globals of the original trainer for one variant: modalities (+extra), seed, V1 or MS patch/pocket bank."""
    spec = {**{k: list(v) for k, v in ORIG_SPEC.items()}, **{m: [m] for m in extra}}
    dims = {m: concat_blocks(ctx.stores['train'], PID0, b).shape[1] for m, b in spec.items()}
    arch = V3['MS'] if ms else V3['V1']
    if ms:
        ms_cls = G[V3['MS_CLASS']]
        factory = {arch: (lambda d: ms_cls(d, mode='attention', latent=G[V3['LATENT']], drop=0.2))}
    else:
        factory = {arch: ORIG_FACTORY[arch]}
    env = {'MODALITY_SPEC': spec, 'DIMS': dims, V3['FACTORY']: factory,
           'SEED': ORIG_SEED if seed is None else seed,
           'BEST_REP_NAME': ORIG_REP_NAME + ''.join(f'+{m}' for m in extra)}
    if ms:
        env.update({V3['RAM']: G[V3['MS_RAM']], V3['COLLATE']: G[V3['MS_COLLATE']]})
    if name is not None:
        env[V3['STATE_ROOT']] = V3_ROOT/'states'/name
    return arch, env

def predict_split(arch, states, split):
    ds = G[V3['DATASET']](split, SPLIT_DF[split])
    preds = []
    for fold in sorted(states):
        model = G[V3['FACTORY']][arch](G['DIMS']).to(DEVICE)
        model.load_state_dict(states[fold])
        p, _ = G[V3['PREDICT']](model, ds)
        preds.append(np.asarray(p, np.float64))
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    return np.mean(np.stack(preds), axis=0)

MEMBERS = {}

def _save_member(name, oof, ext, info):
    np.savez(PRED_DIR/f'{name}.npz', oof=oof, **ext, oof_sha=hashlib.sha1(np.ascontiguousarray(oof)).hexdigest())
    MEMBERS[name] = {'oof': oof, **ext, **info}

def _cached_ext(name, oof):
    fp = PRED_DIR/f'{name}.npz'
    if fp.exists():
        z = np.load(fp, allow_pickle=False)
        if str(z['oof_sha']) == hashlib.sha1(np.ascontiguousarray(oof)).hexdigest() and all(s in z.files for s in SPLITS_EXT):
            return {s: z[s] for s in SPLITS_EXT}
    return None

def run_member(name, ms=False, extra=(), seed=None):
    """Official 5-fold Dev CV with the ORIGINAL trainer (per-fold resume from Drive), then Test300/46 = mean of folds."""
    arch, env = env_for(ms, extra, seed, name)
    with swapped(**env):
        summ, folds, oof, states = G[V3['RUN_CV']](arch)
        oof = np.asarray(oof, np.float64)
        ext = _cached_ext(name, oof) or {s: predict_split(arch, states, s) for s in SPLITS_EXT}
    _save_member(name, oof, ext, {'ms': ms, 'extra': tuple(extra), 'seed': env['SEED']})
    print(f'{name}: Dev fold-mean AUPRC {fold_mean(oof):.6f} | Test300 {average_precision_score(Y["TestSet300"], ext["TestSet300"]):.4f} '
          f'| Test46 {average_precision_score(Y["TestSetNew46"], ext["TestSetNew46"]):.4f}')
    return MEMBERS[name]

# ---------- bootstrap (fast weighted metrics == sklearn with sample_weight; checked below) ----------
def _curve(y, w, order, last):
    ys, ws = y[order], w[order]
    return np.cumsum(ws * ys)[last], np.cumsum(ws * (1 - ys))[last]

def prep(p):
    order = np.argsort(-p, kind='mergesort')
    ps = p[order]
    return order, np.r_[np.flatnonzero(np.diff(ps) != 0), len(ps) - 1]

def w_auprc(y, w, pr):
    tp, fp = _curve(y, w, *pr)
    return float(np.sum(np.diff(np.r_[0.0, tp / tp[-1]]) * tp / np.maximum(tp + fp, 1e-300)))

def w_auroc(y, w, pr):
    tp, fp = _curve(y, w, *pr)
    tpr, fpr = np.r_[0.0, tp / tp[-1]], np.r_[0.0, fp / fp[-1]]
    return float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2))

def w_f1_mcc(y, w, pred):
    tp = np.sum(w * y * pred); fp = np.sum(w * (1 - y) * pred)
    fn = np.sum(w * y * (1 - pred)); tn = np.sum(w * (1 - y) * (1 - pred))
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return float(2 * tp / max(2 * tp + fp + fn, 1e-300)), float((tp * tn - fp * fn) / den) if den > 0 else 0.0

def _metrics(y, w, pr, pred):
    f1, mcc = w_f1_mcc(y, w, pred)
    return {'AUPRC': w_auprc(y, w, pr), 'AUROC': w_auroc(y, w, pr), 'F1@DevThr': f1, 'MCC@DevThr': mcc}

def paired_bootstrap(y, p_ref, p_new, thr_ref, thr_new, prot, B=B_BOOT, seed=BOOT_SEED):
    """Resample proteins with replacement (same proteins for both models)."""
    y = y.astype(float)
    n = int(prot.max()) + 1
    pr_r, pr_n = prep(p_ref), prep(p_new)
    pd_r, pd_n = (p_ref >= thr_ref).astype(float), (p_new >= thr_new).astype(float)
    one = np.ones_like(y)
    pt_r, pt_n = _metrics(y, one, pr_r, pd_r), _metrics(y, one, pr_n, pd_n)
    rng = np.random.default_rng(seed)
    R, N = {k: [] for k in pt_r}, {k: [] for k in pt_r}
    for _ in range(B):
        w = np.bincount(rng.integers(0, n, n), minlength=n)[prot].astype(float)
        if np.sum(w * y) == 0:
            continue
        mr, mn = _metrics(y, w, pr_r, pd_r), _metrics(y, w, pr_n, pd_n)
        for k in pt_r:
            R[k].append(mr[k]); N[k].append(mn[k])
    rows = []
    for k in pt_r:
        r, nw = np.asarray(R[k]), np.asarray(N[k])
        d = nw - r
        rows.append({'metric': k, 'ref': pt_r[k], 'ref_CI95': f'[{np.percentile(r, 2.5):.4f}, {np.percentile(r, 97.5):.4f}]',
                     'new': pt_n[k], 'new_CI95': f'[{np.percentile(nw, 2.5):.4f}, {np.percentile(nw, 97.5):.4f}]',
                     'delta': pt_n[k] - pt_r[k], 'delta_CI95_lo': np.percentile(d, 2.5), 'delta_CI95_hi': np.percentile(d, 97.5),
                     'P(new>ref)': float(np.mean(d > 0)), 'n_boot': len(d)})
    return pd.DataFrame(rows)

def _pooled(a, b):
    return np.concatenate([a, b])

def compare(ref_name, ref, new_name, new, sets=('DevSet1010', 'TestSet300', 'TestSetNew46', 'POOLED_TEST')):
    """ref/new: dict with oof, TestSet300, TestSetNew46, thr (Dev-OOF MCC threshold). Saves + returns the table."""
    out = []
    for s in sets:
        if s == 'DevSet1010':
            y, pr, pn, prot = Y[s], ref['oof'], new['oof'], PROT[s]
        elif s == 'POOLED_TEST':
            y = _pooled(Y['TestSet300'], Y['TestSetNew46'])
            pr, pn = _pooled(ref['TestSet300'], ref['TestSetNew46']), _pooled(new['TestSet300'], new['TestSetNew46'])
            prot = _pooled(PROT['TestSet300'], PROT['TestSetNew46'] + PROT['TestSet300'].max() + 1)
        else:
            y, pr, pn, prot = Y[s], ref[s], new[s], PROT[s]
        t0 = time.time()
        r = paired_bootstrap(y, pr, pn, ref['thr'], new['thr'], prot)
        r.insert(0, 'dataset', s)
        r.insert(0, 'new_model', new_name)
        r.insert(0, 'ref_model', ref_name)
        out.append(r)
        print(f'  bootstrap {new_name} vs {ref_name} | {s}: {time.time() - t0:.0f}s')
    out = pd.concat(out, ignore_index=True)
    out.to_csv(V3_ROOT/f'BOOT_{new_name}_vs_{ref_name}.csv', index=False)
    return out

def show_auprc(tab):
    t = tab[tab.metric.eq('AUPRC')][['dataset', 'ref', 'new', 'delta', 'delta_CI95_lo', 'delta_CI95_hi', 'P(new>ref)']]
    display(t.round(4))
    return t

# the fast metrics must equal sklearn (with and without bootstrap weights)
_y, _p = Y['TestSet300'].astype(float), np.random.default_rng(0).random(len(Y['TestSet300']))
_w = np.bincount(np.random.default_rng(1).integers(0, 300, 300), minlength=300)[PROT['TestSet300']].astype(float)
for _ww in (np.ones_like(_y), _w):
    assert abs(w_auprc(_y, _ww, prep(_p)) - average_precision_score(_y, _p, sample_weight=_ww)) < 1e-9
    assert abs(w_auroc(_y, _ww, prep(_p)) - roc_auc_score(_y, _p, sample_weight=_ww)) < 1e-9
print('V3 setup: PASS | task', V3['TASK'], '| original modalities', ORIG_SPEC, '| seed', ORIG_SEED)


In [ ]:
# ============================================================
# V3-2. Current final model (C0) = the original weights, re-evaluated + bootstrap CI
#       V1 / MS OOF come from the saved fold predictions -> Dev is exactly the original.
# ============================================================
w0 = float(np.linspace(0.0, 1.0, 21)[int(round(V3['ORIG_W_V1'] * 20))])   # exact grid value of the original search
v1_oof = np.asarray(G[V3['V1_OOF']][V3['V1']], np.float64)
ms_oof_ = np.asarray(G[V3['MS_OOF']], np.float64)

arch, env = env_for(ms=False)
with swapped(**env):
    v1_ext = _cached_ext('V1_s0', v1_oof) or {s: predict_split(arch, G[V3['V1_STATES']][V3['V1']], s) for s in SPLITS_EXT}
_save_member('V1_s0', v1_oof, v1_ext, {'ms': False, 'extra': (), 'seed': ORIG_SEED})
arch, env = env_for(ms=True)
with swapped(**env):
    ms_ext = _cached_ext('MS_s0', ms_oof_) or {s: predict_split(arch, G[V3['MS_STATES']], s) for s in SPLITS_EXT}
_save_member('MS_s0', ms_oof_, ms_ext, {'ms': True, 'extra': (), 'seed': ORIG_SEED})

def combo(names, weights):
    out = {k: sum(w * MEMBERS[n][k] for n, w in zip(names, weights)) for k in ('oof', *SPLITS_EXT)}
    out['thr'] = best_mcc_threshold(Y['DevSet1010'], out['oof'])[0]   # Dev-OOF MCC threshold (original convention)
    out['members'], out['weights'] = list(names), [float(w) for w in weights]
    return out

CANDS = {'C0_current': combo(['V1_s0', 'MS_s0'], [w0, 1 - w0])}
C0 = CANDS['C0_current']
print(f"C0 Dev fold-mean AUPRC {fold_mean(C0['oof']):.10f} (original {V3['ORIG_DEV']:.10f})")
assert abs(fold_mean(C0['oof']) - V3['ORIG_DEV']) < 1e-9, 'Dev of the current final is not the original one'
assert abs(fold_mean(v1_oof) - V3['ORIG_V1_DEV']) < 1e-9
for s in SPLITS_EXT:
    a = average_precision_score(Y[s], C0[s])
    print(f"C0 {s}: AUPRC {a:.6f} (original {V3['ORIG_EXT'][s]:.6f}, diff {a - V3['ORIG_EXT'][s]:+.1e})")

# CI of the current final alone (bootstrap of C0 against itself gives its own CI columns)
C0_CI = compare('C0_current', C0, 'C0_current', C0)
display(C0_CI[['dataset', 'metric', 'ref', 'ref_CI95']].round(4))


In [ ]:
# ============================================================
# V3-3. New per-residue features (cached on Drive, attached to the stores under NEW block names)
#   ESMIF_EF : ESM-IF1 encoder (512-d) on the same ESMFold chain-A structures (computed by the metal V4-FIX run)
#   PROTT5   : ProtT5-XL-U50 half-precision encoder (1024-d), the embedding behind bindEmbed21DL / bindNode24
# ============================================================
ESMIF_ROOT = BASE/'cache'/'gear_metal_v3'/'esmif_esmfold_v1'
PROTT5_ROOT = BASE/'cache'/'prott5_xl_half_v1'
PROTT5_MODEL = 'Rostlab/prot_t5_xl_half_uniref50-enc'

def attach(split, pid, block, arr):
    rec = ctx.stores[STORE_KEY[split]][str(pid)]
    tgt = rec['features'] if isinstance(rec, dict) and isinstance(rec.get('features'), dict) else rec
    tgt[block] = arr   # kept float16 (concat_blocks casts per protein) -> half the RAM

def _ok(fp, L, d):
    try:
        a = np.load(fp, mmap_mode='r')
        return a.shape == (L, d) and bool(np.isfinite(a).all())
    except Exception:
        return False

# --- ESM-IF
n_esmif = 0
for split, df in SPLIT_DF.items():
    for r in df.itertuples():
        fp = ESMIF_ROOT/split/'esm_if'/f'{r.prot_id}.npy'
        assert _ok(fp, len(r.sequence), 512), f'missing ESM-IF {split}/{r.prot_id} (run the metal V4 notebook first)'
        attach(split, r.prot_id, 'ESMIF_EF', np.load(fp).astype(np.float16))
        n_esmif += 1
print('ESMIF_EF attached:', n_esmif)

# --- ProtT5 (computed once, ~10 min on a T4)
todo = [(s, str(r.prot_id), r.sequence) for s, df in SPLIT_DF.items() for r in df.itertuples()
        if not _ok(PROTT5_ROOT/s/f'{r.prot_id}.npy', len(r.sequence), 1024)]
print('ProtT5 to compute:', len(todo))
if todo:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'protobuf'], check=True)
    from transformers import T5EncoderModel
    try:   # official ProtT5 usage: space-separated residues, one token per residue + </s>
        from transformers import T5Tokenizer
        _tok = T5Tokenizer.from_pretrained(PROTT5_MODEL, do_lower_case=False)
        def t5_ids(clean):
            ids = _tok(' '.join(clean), add_special_tokens=True)['input_ids']
            if len(ids) != len(clean) + 1:   # residues merged -> map residue pieces explicitly
                ids = _tok.convert_tokens_to_ids([f'\u2581{a}' for a in clean]) + [_tok.eos_token_id]
            return ids, _tok.unk_token_id
    except Exception as e:   # transformers without a usable T5 tokenizer -> the same SentencePiece model directly
        print('T5Tokenizer unavailable -> SentencePiece:', repr(e))
        import sentencepiece as spm
        from huggingface_hub import hf_hub_download
        _sp = spm.SentencePieceProcessor(model_file=hf_hub_download(PROTT5_MODEL, 'spiece.model'))
        def t5_ids(clean):
            return [_sp.piece_to_id(f'\u2581{a}') for a in clean] + [_sp.eos_id()], _sp.unk_id()
    t5 = T5EncoderModel.from_pretrained(PROTT5_MODEL).to(DEVICE).eval()
    t5 = t5.half() if DEVICE.type == 'cuda' else t5.float()   # Rostlab usage: half precision on GPU
    for n, (s, pid, seq) in enumerate(sorted(todo, key=lambda x: len(x[2])), start=1):
        clean = re.sub(r'[UZOB]', 'X', seq)
        ids, unk = t5_ids(clean)
        assert len(ids) == len(seq) + 1 and unk not in ids[:-1], (pid, len(ids), len(seq))
        ids = torch.tensor(ids, dtype=torch.long)
        with torch.no_grad():
            h = t5(input_ids=ids[None].to(DEVICE), attention_mask=torch.ones_like(ids)[None].to(DEVICE)).last_hidden_state
        arr = h[0, :len(seq)].float().cpu().numpy().astype(np.float16)
        assert np.isfinite(arr).all(), pid
        fp = PROTT5_ROOT/s/f'{pid}.npy'
        fp.parent.mkdir(parents=True, exist_ok=True)
        tmp = fp.with_name(f'.tmp.{fp.name}')
        np.save(tmp, arr)
        tmp.replace(fp)
        if n % 200 == 0:
            print('ProtT5', n, '/', len(todo))
    del t5
    torch.cuda.empty_cache()
for split, df in SPLIT_DF.items():
    for r in df.itertuples():
        attach(split, r.prot_id, 'PROTT5', np.load(PROTT5_ROOT/split/f'{r.prot_id}.npy'))
print('PROTT5 attached:', sum(len(df) for df in SPLIT_DF.values()))


In [ ]:
# ============================================================
# V3-4. Phase A — new modality on the ORIGINAL V1 hypergraph (same trainer, same seed), Dev-only decision
#       Each variant: paired fold test vs V1 + protein bootstrap (B = 10000) vs V1.
# ============================================================
from scipy.stats import ttest_rel

PHASE_A = [('ESMIF_EF',), ('PROTT5',), ('ESMIF_EF', 'PROTT5')]
suffix = lambda extra: ''.join(f'+{m}' for m in extra)
V1 = MEMBERS['V1_s0']
V1['thr'] = best_mcc_threshold(Y['DevSet1010'], V1['oof'])[0]
rows, BOOT_A = [], []
for extra in PHASE_A:
    name = f'V1{suffix(extra)}_s0'
    m = run_member(name, extra=extra)
    m['thr'] = best_mcc_threshold(Y['DevSet1010'], m['oof'])[0]
    fa, fb = fold_aps(V1['oof']), fold_aps(m['oof'])
    rows.append({'variant': name, 'extra': extra, 'Dev_fold_mean': fb.mean(), 'delta_vs_V1': fb.mean() - fa.mean(),
                 'folds_up': int((fb > fa).sum()), 'paired_t_p': float(ttest_rel(fb, fa).pvalue)})
    tab = compare('V1_s0', V1, name, m)
    BOOT_A.append(tab)
    show_auprc(tab)
phase_a = pd.DataFrame(rows).sort_values('Dev_fold_mean', ascending=False).reset_index(drop=True)
phase_a.to_csv(V3_ROOT/'PHASE_A_modalities.csv', index=False)
display(phase_a.round(4))
best = phase_a.iloc[0]
BEST_MOD = tuple(best.extra) if best.delta_vs_V1 >= MARGIN else ()
print('PHASE A decision (Dev only):', f'{best.variant} (+{best.delta_vs_V1:.4f})' if BEST_MOD else 'no modality beats V1 by the margin')


In [ ]:
# ============================================================
# V3-5. Phase B — MS hypergraph with the chosen modality; Phase C — 2 extra seeds for the final pair
#       (seed bagging: mean of 3 independently trained 5-fold models per member)
# ============================================================
PAIR = ('V1' + suffix(BEST_MOD), 'MS' + suffix(BEST_MOD))   # V1<mods>_s0 already trained in Phase A
if BEST_MOD:
    run_member(f'{PAIR[1]}_s0', ms=True, extra=BEST_MOD)
EXTRA_SEEDS = (ORIG_SEED + 1000, ORIG_SEED + 2000)
for k, sd in enumerate(EXTRA_SEEDS, start=1):
    run_member(f'{PAIR[0]}_s{k}', ms=False, extra=BEST_MOD, seed=sd)
    run_member(f'{PAIR[1]}_s{k}', ms=True, extra=BEST_MOD, seed=sd)
for base in PAIR:
    names = [f'{base}_s{k}' for k in range(len(EXTRA_SEEDS) + 1)]
    bag = {k: np.mean([MEMBERS[n][k] for n in names], axis=0) for k in ('oof', *SPLITS_EXT)}
    MEMBERS[f'{base}_bag3'] = bag
    print(f'{base}: seeds {[round(fold_mean(MEMBERS[n]["oof"]), 4) for n in names]} -> bag3 {fold_mean(bag["oof"]):.4f}')


In [ ]:
# ============================================================
# V3-6. Candidate final models, Dev-only decision, then external + bootstrap (B = 10000) vs the current final
#   weights: grid on Dev (step 0.05 for pairs, 0.1 for 4 members) maximising the official 5-fold mean AUPRC
#   decision score: CROSS-FITTED Dev (weights chosen on 4 folds, scored on the 5th) -> no optimism from the search
#   rule (fixed now): simplest first; a candidate replaces the winner only if cross-fitted gain >= 0.002
# ============================================================
from itertools import product

def simplex(k, step):
    n = int(round(1 / step))
    return [np.array(c) / n for c in product(range(n + 1), repeat=k) if sum(c) == n]

def search(names, step, folds=None):
    folds = list(FOLD_IDX) if folds is None else folds
    P = [MEMBERS[n]['oof'] for n in names]
    best_w, best_s = None, -1
    for w in simplex(len(names), step):
        p = sum(wi * pi for wi, pi in zip(w, P))
        s = np.mean([average_precision_score(Y['DevSet1010'][FOLD_IDX[f]], p[FOLD_IDX[f]]) for f in folds])
        if s > best_s + 1e-12:
            best_w, best_s = w, s
    return best_w, best_s

def crossfit(names, step):
    P = [MEMBERS[n]['oof'] for n in names]
    sc = []
    for f in FOLD_IDX:
        w, _ = search(names, step, [g for g in FOLD_IDX if g != f])
        p = sum(wi * pi for wi, pi in zip(w, P))
        sc.append(average_precision_score(Y['DevSet1010'][FOLD_IDX[f]], p[FOLD_IDX[f]]))
    return float(np.mean(sc))

SPECS = [('C0_current', ['V1_s0', 'MS_s0'], 0.05)]
if BEST_MOD:
    SPECS.append(('C1_newmod_pair', [f'{PAIR[0]}_s0', f'{PAIR[1]}_s0'], 0.05))
SPECS.append(('C2_bagged_pair', [f'{PAIR[0]}_bag3', f'{PAIR[1]}_bag3'], 0.05))
if BEST_MOD:
    SPECS.append(('C3_union', ['V1_s0', 'MS_s0', f'{PAIR[0]}_bag3', f'{PAIR[1]}_bag3'], 0.1))

rows = []
for cname, names, step in SPECS:
    w, s_in = search(names, step)
    s_cf = crossfit(names, step)
    if cname == 'C0_current':
        w, s_in = np.array([w0, 1 - w0]), fold_mean(C0['oof'])   # keep the original final exactly
    else:
        CANDS[cname] = combo(names, w)
    rows.append({'candidate': cname, 'members': ' + '.join(names), 'weights': np.round(w, 3).tolist(),
                 'Dev_in_sample': s_in, 'Dev_crossfit': s_cf})
dec = pd.DataFrame(rows)
win = dec.iloc[0]
for _, r in dec.iloc[1:].iterrows():
    if r.Dev_crossfit - win.Dev_crossfit >= MARGIN:
        win = r
dec['crossfit_gain_vs_C0'] = dec.Dev_crossfit - dec.Dev_crossfit.iloc[0]
dec.to_csv(V3_ROOT/'CANDIDATES_DEV.csv', index=False)
display(dec.round(4))
WINNER = str(win.candidate)
print('DEV-ONLY DECISION:', WINNER)

BOOT_C = []
for cname in CANDS:
    if cname == 'C0_current':
        continue
    print('\n' + cname)
    tab = compare('C0_current', C0, cname, CANDS[cname])
    BOOT_C.append(tab)
    show_auprc(tab)
BOOT_ALL = pd.concat([C0_CI, *BOOT_A, *BOOT_C], ignore_index=True)
BOOT_ALL.to_csv(V3_ROOT/'BOOTSTRAP_ALL_B10000.csv', index=False)

summary = []
for cname, c in CANDS.items():
    summary.append({'candidate': cname, 'Dev_fold_mean': fold_mean(c['oof']),
                    **{f'{s}_AUPRC': average_precision_score(Y[s], c[s]) for s in SPLITS_EXT},
                    'POOLED_AUPRC': average_precision_score(_pooled(Y['TestSet300'], Y['TestSetNew46']),
                                                            _pooled(c['TestSet300'], c['TestSetNew46'])),
                    'Dev_MCC_threshold': c['thr']})
summary = pd.DataFrame(summary)
summary.to_csv(V3_ROOT/'CANDIDATES_EXTERNAL.csv', index=False)
display(summary.round(4))

final = WINNER
if WINNER != 'C0_current':
    t = [b for b in BOOT_C if b.new_model.iloc[0] == WINNER][0]
    pr = t[(t.dataset == 'POOLED_TEST') & (t.metric == 'AUPRC')].iloc[0]
    print(f"\n{WINNER} vs C0, pooled test AUPRC: {pr.delta:+.4f} CI95 [{pr.delta_CI95_lo:+.4f}, {pr.delta_CI95_hi:+.4f}] "
          f"P={pr['P(new>ref)']:.3f}")
    if pr.delta_CI95_hi < 0:
        final = 'C0_current'
        print('STOP RULE: significantly worse on the tests -> keep the current final')
    elif pr.delta_CI95_lo > 0:
        print('VERDICT: significantly better on the tests -> new final =', WINNER)
    else:
        print('VERDICT: not significant on the tests -> the Dev decision stands: new final =', WINNER)
else:
    print('VERDICT: nothing beats the current final on Dev -> keep C0')
FINAL = CANDS[final]
(V3_ROOT/'FINAL_DECISION.json').write_text(json.dumps({
    'task': V3['TASK'], 'final': final, 'dev_only_winner': WINNER, 'members': FINAL['members'], 'weights': FINAL['weights'],
    'dev_mcc_threshold': FINAL['thr'], 'best_modality': list(BEST_MOD), 'bootstrap_B': B_BOOT,
    'candidates_dev': dec.to_dict('records')}, indent=2, default=str))
print('FINAL:', final, FINAL['members'], FINAL['weights'])


In [ ]:
# ============================================================
# V3-7. Export -> download, unzip into protein-binding/data/ (paths relative to Protein-binding-Phi/)
# ============================================================
import zipfile
OUT = BASE/'EXPORT'
OUT.mkdir(exist_ok=True)
zp = OUT/f"{V3['TASK'].lower()}_v3_improve.zip"
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(V3_ROOT.rglob('*')):   # member predictions, new member states, bootstrap + decision tables
        if p.is_file():
            z.write(p, p.relative_to(BASE))
print('zip:', zp, f'{zp.stat().st_size/1e6:.1f} MB')
EXPORT_PROTT5 = 'PROTT5' in BEST_MOD   # ProtT5 features are only needed locally when the final model uses them
if EXPORT_PROTT5:
    zf = OUT/'prott5_xl_half_v1_features.zip'
    if not zf.exists():
        with zipfile.ZipFile(zf, 'w', zipfile.ZIP_STORED) as z:
            for p in sorted(PROTT5_ROOT.rglob('*.npy')):
                z.write(p, p.relative_to(BASE))
    print('zip:', zf, f'{zf.stat().st_size/1e6:.1f} MB')


# Part 2 — DNA model line


In [ ]:
# ============================================================
# LX-0. Configuration of the dna model line
#   Prerequisite: every cell above has run (part 1 = notebook 09), so MEMBERS / CANDS / the V3 helpers exist.
# ============================================================
TARGET = 'dna'
LX_SEEDS = (0, 1, 2)                      # seed bagging, as in notebook 09
LX_ROOT = BASE/'results'/'NUCLEAR_DNA_LINE'
LX_ROOT.mkdir(parents=True, exist_ok=True)

# BioLiP2 snapshot 2026-09-29 (resolution <= 3 A), ligand_id dna/rna per UniProt accession.
# D = DNA, R = RNA, B = both, U = not found in BioLiP (masked).
LX_KIND_JSON = r"""{"A4FTK7":"D","B3LQW9":"U","B8FW11":"D","B8H4R9":"D","C1IFD2":"R","C3SZN7":"D","C3W947":"D","C8V0B5":"D","D0CAL0":"R","D0CBZ8":"R","D0CDQ7":"R","D5MNX7":"D","D6E269":"U","F0NDX5":"R","G2RUZ1":"D","L7IX95":"D","O15527":"D","O25841":"D","O28416":"B","O28417":"D","O29975":"D","O35013":"R","O43809":"R","O52512":"D","O59188":"D","O74859":"D","O87963":"D","O94408":"R","P00648":"D","P00651":"R","P00698":"R","P01127":"D","P02263":"D","P02281":"D","P02958":"D","P03004":"D","P03013":"D","P03018":"D","P03034":"D","P03036":"D","P03042":"D","P03050":"D","P03052":"D","P03206":"D","P03496":"R","P03536":"R","P03612":"R","P03697":"D","P03699":"D","P03882":"D","P04390":"D","P04392":"D","P04418":"D","P06134":"D","P06534":"D","P06897":"D","P06956":"D","P09132":"R","P09184":"D","P0A0I7":"D","P0A459":"D","P0A6C1":"D","P0A6X7":"D","P0A776":"R","P0A7D1":"R","P0A881":"D","P0A8J2":"D","P0A8U6":"D","P0A988":"D","P0ACH5":"D","P0ACP7":"D","P0ACT4":"D","P0AEK0":"D","P0AF28":"D","P0AFJ5":"D","P0AG51":"R","P0AGB6":"D","P0CI76":"D","P0CY08":"D","P10828":"D","P10970":"R","P10971":"R","P10972":"R","P11690":"R","P11746":"D","P12732":"R","P12733":"R","P12734":"R","P12737":"R","P13123":"D","P13393":"D","P14116":"R","P14119":"R","P14121":"R","P14135":"R","P16458":"D","P18138":"R","P19793":"D","P22450":"R","P22452":"R","P23657":"D","P23873":"D","P24321":"U","P32410":"R","P32411":"R","P32761":"D","P33905":"D","P34257":"D","P37108":"R","P39075":"D","P44558":"D","P44688":"D","P49458":"R","P50187":"D","P50465":"D","P56178":"D","P56930":"R","P60618":"R","P60619":"R","P60848":"R","P62661":"U","P62669":"R","P62799":"D","P62801":"D","P62805":"D","P63159":"D","P67809":"R","P68919":"R","P69202":"D","P69687":"R","P71650":"R","P76632":"D","P80377":"R","P80382":"U","P84131":"D","P84229":"D","P84233":"D","Q03243":"R","Q05783":"D","Q14493":"R","Q2G285":"D","Q32WH4":"D","Q37964":"D","Q45488":"D","Q46864":"D","Q46898":"B","Q4UNB3":"D","Q57468":"D","Q5FBS0":"D","Q5SHN3":"R","Q5SHP2":"R","Q5SJ65":"D","Q5SJ76":"R","Q5SJH3":"R","Q5SLP8":"R","Q5Y812":"D","Q64368":"R","Q6NS38":"D","Q74ML9":"B","Q7LFL8":"D","Q7MHK3":"D","Q8C6P8":"D","Q8DQG2":"D","Q8GGH0":"D","Q8IKH2":"D","Q8NMG3":"D","Q8QL27":"R","Q8SXK5":"D","Q8WQM6":"D","Q8YYB7":"U","Q8Z2A5":"D","Q95VR4":"D","Q97ZE3":"D","Q9BTM1":"D","Q9BTP7":"D","Q9DHS8":"D","Q9HJY3":"D","Q9HSF4":"D","Q9I0M3":"D","Q9KJ88":"D","Q9KXR9":"D","Q9LSX0":"D","Q9LW31":"D","Q9X7H4":"D"}"""
print('LX config: target', TARGET, '| seeds', LX_SEEDS, '| root', LX_ROOT)


In [ ]:
# ============================================================
# LX-1. Target labels for this model line (TARGET = 'dna' or 'rna')
#
# The per-residue nuclear labels are never changed. BioLiP2 only says which nucleic acid each protein binds:
#       protein binds TARGET (or both) -> y = its nuclear labels        other nuclear protein -> y = 0
#       protein has no nuclear site    -> y = 0 (a true negative)       not found in BioLiP   -> MASKED
# Masked = dropped from the loss and from every metric below (6 proteins in Dev, none in the test sets), because
# their nuclear residues are DNA or RNA but we do not know which: scoring them either way would be invented data.
# ============================================================
KIND = {k: {'D': 'dna', 'R': 'rna', 'B': 'both', 'U': 'unknown'}[v] for k, v in json.loads(LX_KIND_JSON).items()}
YT, MASK, LXY, LXM = {}, {}, {}, {}          # per split arrays + per protein lookups used by the trainer
rows = []
for split, df in SPLIT_DF.items():
    ys, ms = [], []
    for r in df.itertuples():
        pid, nuc = str(r.prot_id), np.asarray(getattr(r, LABEL), np.float32)
        kind = KIND.get(pid, 'unknown') if nuc.sum() else 'negative'
        y = nuc if kind in (TARGET, 'both') else np.zeros_like(nuc)
        m = np.zeros_like(nuc) if kind == 'unknown' else np.ones_like(nuc)
        LXY[split, pid], LXM[split, pid] = y, m
        ys.append(y); ms.append(m)
        rows.append({'split': split, 'kind': kind, 'positives': int(y.sum()), 'masked': int(len(nuc) - m.sum())})
    YT[split] = np.concatenate(ys).astype(np.int64)
    MASK[split] = np.concatenate(ms).astype(bool)

audit = pd.DataFrame(rows).groupby(['split', 'kind']).agg(proteins=('kind', 'size'), positives=('positives', 'sum'),
                                                          masked_residues=('masked', 'sum'))
display(audit)
LX_FOLD = {f: idx[MASK['DevSet1010'][idx]] for f, idx in FOLD_IDX.items()}       # masked official folds
KEEP = {s: np.flatnonzero(MASK[s]) for s in SPLIT_DF}                            # residues kept per split
PROT_K = {s: pd.factorize(PROT[s][KEEP[s]])[0] for s in SPLIT_DF}                # protein index after masking

def lx_fold_aps(p):
    return np.array([average_precision_score(YT['DevSet1010'][i], np.asarray(p)[i]) for i in LX_FOLD.values()])

def lx_fold_mean(p):
    return float(lx_fold_aps(p).mean())

def lx_ap(split, p):
    k = KEEP[split]
    return float(average_precision_score(YT[split][k], np.asarray(p)[k]))

print(f"TARGET = {TARGET} | Dev positives {int(YT['DevSet1010'].sum())} "
      f"(nuclear total {int(Y['DevSet1010'].sum())}) | masked residues {int((~MASK['DevSet1010']).sum())}")
for s in ('TestSet300', 'TestSetNew46'):
    print(f"   {s}: positives {int(YT[s].sum())} / nuclear {int(Y[s].sum())}, masked {int((~MASK[s]).sum())}")


In [ ]:
# ============================================================
# LX-2. The model to beat: the CURRENT nuclear final, scored against this target.
#       It is a nucleic-acid model, so it already finds many of these residues. A separate model line is only
#       worth it if it beats this on Dev.
# ============================================================
C0_FINAL = CANDS['C3_union']
C3_TERMS = [{'members': [m], 'weight': w} for m, w in zip(C0_FINAL['members'], C0_FINAL['weights'])]
NUCLEAR_AS_TARGET = {k: sum(t['weight'] * MEMBERS[t['members'][0]][k] for t in C3_TERMS if t['weight'])
                     for k in ('oof', *SPLITS_EXT)}
MEMBERS['nuclear_final'] = NUCLEAR_AS_TARGET
base_dev = lx_fold_mean(NUCLEAR_AS_TARGET['oof'])
print(f"nuclear final scored on {TARGET}: Dev fold-mean {base_dev:.4f} | "
      f"Test300 {lx_ap('TestSet300', NUCLEAR_AS_TARGET['TestSet300']):.4f} | "
      f"Test46 {lx_ap('TestSetNew46', NUCLEAR_AS_TARGET['TestSetNew46']):.4f}")


In [ ]:
# ============================================================
# LX-3. The model line: the same hypergraphs (V1 broad patches, MS multi-scale patches) with the same modalities
#       (ESM + ACCESS + patch blocks + ESM-IF + ProtT5), trained on THIS target only, masked loss, 3 seeds.
#       Resume-safe: every fold is saved on Drive, so a disconnect costs nothing.
# ============================================================
class TargetDataset(PatchProteinDataset):
    def __getitem__(self, i):
        d = super().__getitem__(i)
        d['y'] = LXY[self.split, d['pid']]
        d['mask'] = LXM[self.split, d['pid']]
        return d


def make_lx_collate(base):
    def collate(batch):
        out = base(batch)
        out['mask'] = torch.from_numpy(np.concatenate([b['mask'] for b in batch])).float()
        return out
    return collate


def make_lx_trainer(base_collate):
    def train_fold(arch, tr_ds, va_ds, fold):
        torch.manual_seed(SEED + 100 + fold)
        np.random.seed(SEED + 100 + fold)
        model = PATCH_FACTORY[arch](DIMS).to(DEVICE)
        ids = [str(p) for p in tr_ds.df.prot_id]
        yt = np.concatenate([LXY['DevSet1010', p] for p in ids])
        mk = np.concatenate([LXM['DevSet1010', p] for p in ids])
        npos = max(float((yt * mk).sum()), 1.0)
        pw = torch.tensor(np.sqrt(max(float(mk.sum()) - npos, 1.0) / npos), device=DEVICE)
        crit = nn.BCEWithLogitsLoss(pos_weight=pw, reduction='none')
        opt = torch.optim.AdamW(model.parameters(), lr=PATCH_LR, weight_decay=PATCH_WEIGHT_DECAY)
        loader = DataLoader(tr_ds, batch_size=PATCH_PROTEIN_BATCH, shuffle=True, num_workers=0,
                            pin_memory=True, collate_fn=make_lx_collate(base_collate))
        best_ap, best_state, bad = -1.0, None, 0
        for epoch in range(1, PATCH_EPOCHS + 1):
            model.train()
            losses = []
            for b in loader:
                m = b['mask'].to(DEVICE)         # move_patch_batch rebuilds the dict with 5 keys only
                b = move_patch_batch(b)
                opt.zero_grad(set_to_none=True)
                logits = model(b['mods'], b['edge_res'], b['edge_patch'], b['patch_feat'])
                loss = (crit(logits, b['y']) * m).sum() / m.sum().clamp_min(1.0)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
                losses.append(float(loss.detach().cpu()))
            vp, vy = predict_patch(model, va_ds)
            val_ap = ap(vy, vp)              # masked proteins are 0.6% of Dev; they only touch early stopping
            if val_ap > best_ap + 1e-05:
                best_ap, bad = val_ap, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
            if epoch == 1 or epoch % 5 == 0:
                print(arch, 'fold', fold, 'epoch', epoch, 'loss', round(float(np.mean(losses)), 4),
                      'val_AP', round(val_ap, 5), 'best', round(best_ap, 5))
            if bad >= PATCH_PATIENCE:
                break
        model.load_state_dict(best_state)
        return model, best_state
    return train_fold


def run_lx_member(name, ms=False, seed=None):
    arch, env = env_for(ms, BEST_MOD, seed, name)
    env['PatchProteinDataset'] = TargetDataset
    env['train_patch_fold'] = make_lx_trainer(collate_patch_ms if ms else collate_patch)
    with swapped(**env):
        _, _, oof, states = run_patch_cv(arch)
        oof = np.asarray(oof, np.float64)
        ext = _cached_ext(name, oof) or {s: predict_split(arch, states, s) for s in SPLITS_EXT}
    _save_member(name, oof, ext, {'ms': ms, 'target': TARGET, 'seed': env['SEED']})
    print(f"{name}: Dev fold-mean {TARGET} AUPRC {lx_fold_mean(oof):.6f} | "
          f"Test300 {lx_ap('TestSet300', ext['TestSet300']):.4f} | Test46 {lx_ap('TestSetNew46', ext['TestSetNew46']):.4f}")
    return MEMBERS[name]


for s in LX_SEEDS:
    for ms in (False, True):
        run_lx_member(f"{TARGET}_{'MS' if ms else 'V1'}_s{s}", ms=ms, seed=None if s == 0 else ORIG_SEED + 1000 * s)
for ms in (False, True):
    tag = 'MS' if ms else 'V1'
    names = [f'{TARGET}_{tag}_s{s}' for s in LX_SEEDS]
    MEMBERS[f'{TARGET}_{tag}_bag'] = {k: np.mean([MEMBERS[n][k] for n in names], axis=0)
                                      for k in ('oof', *SPLITS_EXT)}
    print(f"{TARGET}_{tag}: seeds {[round(lx_fold_mean(MEMBERS[n]['oof']), 4) for n in names]} -> "
          f"bag {lx_fold_mean(MEMBERS[f'{TARGET}_{tag}_bag']['oof']):.4f}")


In [ ]:
# ============================================================
# LX-4. Dev-only decision. Weights are searched on Dev and scored CROSS-FITTED (searched on 4 folds, scored on the
#       5th), so the comparison is not inflated by the search. A candidate wins only by >= 0.002.
# ============================================================
from itertools import product

def lx_search(names, step, folds=None):
    folds = list(LX_FOLD) if folds is None else folds
    P = [np.asarray(MEMBERS[n]['oof']) for n in names]
    n = int(round(1 / step))
    best_w, best_s = None, -1.0
    for c in product(range(n + 1), repeat=len(names)):
        if sum(c) != n:
            continue
        w = np.array(c) / n
        p = sum(wi * pi for wi, pi in zip(w, P))
        s = float(np.mean([average_precision_score(YT['DevSet1010'][LX_FOLD[f]], p[LX_FOLD[f]]) for f in folds]))
        if s > best_s + 1e-12:
            best_w, best_s = w, s
    return best_w, best_s


def lx_crossfit(names, step):
    P = [np.asarray(MEMBERS[n]['oof']) for n in names]
    out = []
    for f in LX_FOLD:
        w, _ = lx_search(names, step, [g for g in LX_FOLD if g != f])
        p = sum(wi * pi for wi, pi in zip(w, P))
        out.append(average_precision_score(YT['DevSet1010'][LX_FOLD[f]], p[LX_FOLD[f]]))
    return float(np.mean(out))


def lx_combo(names, weights):
    out = {k: sum(w * np.asarray(MEMBERS[n][k]) for n, w in zip(names, weights)) for k in ('oof', *SPLITS_EXT)}
    out['thr'] = best_mcc_threshold(YT['DevSet1010'][KEEP['DevSet1010']], out['oof'][KEEP['DevSet1010']])[0]
    out['members'], out['weights'] = list(names), [float(w) for w in weights]
    return out


SPECS = [('B0_nuclear_final', ['nuclear_final'], 1.0),
         (f'S1_{TARGET}_pair', [f'{TARGET}_V1_bag', f'{TARGET}_MS_bag'], 0.05),
         (f'S2_{TARGET}_plus_nuclear', ['nuclear_final', f'{TARGET}_V1_bag', f'{TARGET}_MS_bag'], 0.1)]
LX_CANDS, rows = {}, []
for cname, names, step in SPECS:
    if len(names) == 1:
        w, s_in = np.array([1.0]), lx_fold_mean(MEMBERS[names[0]]['oof'])
        s_cf = s_in
    else:
        w, s_in = lx_search(names, step)
        s_cf = lx_crossfit(names, step)
    LX_CANDS[cname] = lx_combo(names, w)
    rows.append({'candidate': cname, 'members': ' + '.join(names), 'weights': np.round(w, 3).tolist(),
                 'Dev_in_sample': s_in, 'Dev_crossfit': s_cf})
dec = pd.DataFrame(rows)
win = dec.iloc[0]
for _, r in dec.iloc[1:].iterrows():
    if r.Dev_crossfit - win.Dev_crossfit >= MARGIN:
        win = r
dec['gain_vs_nuclear_final'] = dec.Dev_crossfit - dec.Dev_crossfit.iloc[0]
dec.to_csv(LX_ROOT/'CANDIDATES_DEV.csv', index=False)
display(dec.round(4))
LX_WINNER = str(win.candidate)
print(f'DEV-ONLY DECISION for {TARGET}:', LX_WINNER)


In [ ]:
# ============================================================
# LX-5. External confirmation + paired protein bootstrap (B = 10000) of every candidate against the nuclear final.
#       Decision rule, fixed now: the Dev winner stands unless the pooled-test CI95 lies entirely below 0.
# ============================================================
def lx_compare(ref_name, ref, new_name, new):
    out = []
    for s, key in (('DevSet1010', 'oof'), ('TestSet300', 'TestSet300'), ('TestSetNew46', 'TestSetNew46'),
                   ('POOLED_TEST', None)):
        if key is None:
            k3, k46 = KEEP['TestSet300'], KEEP['TestSetNew46']
            y = np.r_[YT['TestSet300'][k3], YT['TestSetNew46'][k46]]
            pr = np.r_[np.asarray(ref['TestSet300'])[k3], np.asarray(ref['TestSetNew46'])[k46]]
            pn = np.r_[np.asarray(new['TestSet300'])[k3], np.asarray(new['TestSetNew46'])[k46]]
            prot = np.r_[PROT_K['TestSet300'], PROT_K['TestSetNew46'] + PROT_K['TestSet300'].max() + 1]
        else:
            k = KEEP[s]
            y, pr, pn, prot = YT[s][k], np.asarray(ref[key])[k], np.asarray(new[key])[k], PROT_K[s]
        r = paired_bootstrap(y, pr, pn, ref['thr'], new['thr'], prot)
        r.insert(0, 'dataset', s)
        r.insert(0, 'new_model', new_name)
        out.append(r)
    return pd.concat(out, ignore_index=True)


summary, boots = [], []
for cname, c in LX_CANDS.items():
    row = {'candidate': cname, 'Dev_fold_mean': lx_fold_mean(c['oof']), 'Dev_thr': c['thr']}
    for s in SPLITS_EXT:
        k = KEEP[s]
        yy, pp = YT[s][k], np.asarray(c[s])[k]
        row[f'{s}_AUPRC'] = float(average_precision_score(yy, pp))
        pred = pp >= c['thr']
        row[f'{s}_precision'] = float(precision_score(yy, pred, zero_division=0))
        row[f'{s}_recall'] = float(recall_score(yy, pred, zero_division=0))
    summary.append(row)
    if cname != 'B0_nuclear_final':
        tab = lx_compare('B0_nuclear_final', LX_CANDS['B0_nuclear_final'], cname, c)
        boots.append(tab)
        print('\n' + cname)
        display(tab[tab.metric.eq('AUPRC')][['dataset', 'ref', 'new', 'delta', 'delta_CI95_lo', 'delta_CI95_hi',
                                             'P(new>ref)']].round(4))
summary = pd.DataFrame(summary)
summary.to_csv(LX_ROOT/'CANDIDATES_EXTERNAL.csv', index=False)
display(summary.round(4))
if boots:
    allb = pd.concat(boots, ignore_index=True)
    allb.to_csv(LX_ROOT/'BOOTSTRAP_B10000.csv', index=False)

final = LX_WINNER
if LX_WINNER != 'B0_nuclear_final':
    t = [b for b in boots if b.new_model.iloc[0] == LX_WINNER][0]
    pr = t[(t.dataset == 'POOLED_TEST') & (t.metric == 'AUPRC')].iloc[0]
    print(f"\n{LX_WINNER} vs nuclear final, pooled test AUPRC: {pr.delta:+.4f} "
          f"CI95 [{pr.delta_CI95_lo:+.4f}, {pr.delta_CI95_hi:+.4f}] P={pr['P(new>ref)']:.3f}")
    if pr.delta_CI95_hi < 0:
        final = 'B0_nuclear_final'
        print('STOP RULE: significantly worse on the tests -> keep the nuclear final for this ligand')
    else:
        print(f'VERDICT: {TARGET} model line =', LX_WINNER)
else:
    print(f'VERDICT: no {TARGET}-specific model beats the nuclear final on Dev -> keep the nuclear final')
LX_FINAL = LX_CANDS[final]
(LX_ROOT/'FINAL_DECISION.json').write_text(json.dumps(
    {'target': TARGET, 'final': final, 'dev_only_winner': LX_WINNER, 'members': LX_FINAL['members'],
     'weights': LX_FINAL['weights'], 'dev_mcc_threshold': LX_FINAL['thr'], 'bootstrap_B': B_BOOT,
     'masked_dev_proteins': int(sum(1 for (s, _p), m in LXM.items() if s == 'DevSet1010' and m.sum() == 0)),
     'candidates': dec.to_dict('records')}, indent=2, default=str))
print('FINAL:', final, LX_FINAL['members'], LX_FINAL['weights'])


In [ ]:
# ============================================================
# LX-6. Export -> download, unzip into protein-binding/data/
# ============================================================
import zipfile
OUT = BASE/'EXPORT'
OUT.mkdir(exist_ok=True)
zp = OUT/f'nuclear_{TARGET}_line.zip'
mine = [n for n in MEMBERS if n.startswith(f'{TARGET}_')]
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(LX_ROOT.rglob('*')):
        if p.is_file():
            z.write(p, p.relative_to(BASE))
    for n in mine:
        fp = PRED_DIR/f'{n}.npz'
        if fp.exists():
            z.write(fp, fp.relative_to(BASE))
        for q in sorted((V3_ROOT/'states'/n).rglob('*')):
            if q.is_file():
                z.write(q, q.relative_to(BASE))
print('zip:', zp, f'{zp.stat().st_size/1e6:.1f} MB |', len(mine), 'members')
